# Reproducibility copy of original Colab notebook
This notebook preserves the authors' analytical workflow, with **saved outputs removed** and credentials/path references sanitized.
It has not been validated end-to-end in a clean environment. Some cells depend on legacy intermediate variables, remote IUCN access, or private screening exports. See `docs/reproducibility.md`.


In [ ]:
# %%capture
!pip install -U kaleido

In [ ]:
import requests
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import plotly.express as px
import seaborn as sns
import numpy as np
import plotly.io as pio
import warnings
import random
from collections import Counter, defaultdict
from scipy.stats import qmc  # For Sobol sequence generation
import json
from plotly.subplots import make_subplots

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning, module='matplotlib.font_manager')
warnings.filterwarnings("ignore")

In [ ]:
# # Set default font family for matplotlib
# plt.rcParams['font.family'] = 'Arial'

# # Set default font family for plotly
# pio.templates.default = "plotly_white"
# pio.templates["plotly_white"].layout.font.family = "Arial"

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
url = ROOT / 'data' / 'processed' / 'study_extraction.csv'


In [ ]:
words_to_remove = [
    "Paper Information",
    "ML Pipeline Information",
    "Dataset Information",
    "Geographic & Taxonomic Information",
    "Performance Information",
    "Synthesis Information"
]

In [ ]:
df = pd.read_csv(url)

In [ ]:
def clean_column(col_name):
    for word in words_to_remove:
        if col_name.startswith(word) and col_name != "Data quality control":
            col_name = col_name.replace(word, "").strip()
    return col_name

df.columns = [clean_column(col) for col in df.columns]

In [ ]:
# Convert numeric fields
numeric_cols = [
    "Input Segment Length Seconds", "Dataset Size Train",
    "Dataset Size Validation", "Dataset Size Testing",
    "Total Duration Hours", "Accuracy", "Precision",
    "Recall", "Specificity", "F1 Score", "UAR"
]
# df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [ ]:
df = df.dropna(subset=['id'])

In [ ]:
df.columns

# Data Extraction

## Literature search and study selection

In [ ]:
studies_per_year_df = df.drop(index=4)
studies_per_year_df = studies_per_year_df['Year'].value_counts().sort_index().reset_index()
studies_per_year_df.columns = ['Year', 'Number of Studies']

# Create the histogram
fig = px.bar(
    studies_per_year_df,
    x='Year',
    y='Number of Studies',
    # title='Number of Studies Published Per Year',
    labels={'Year': 'Year of Publication', 'Number of Studies': 'Number of Studies'}
)

# Increase font size
fig.update_layout(
    font=dict(size=18), # applies to everything: titles, labels, legend, etc.
    xaxis=dict(title=dict(font=dict(size=23)), tickfont=dict(size=18)),
    yaxis=dict(title=dict(font=dict(size=23)), tickfont=dict(size=18))
)

# fig.write_image("studies_per_year_histogram.png")
# fig.write_image("studies_per_year_histogram.pdf")

fig.show()

## Taxonomic coverage

In [ ]:
years = []
taxa_orders = []
species_list = []

for index, row in df.iterrows():
    year = row['Year']
    species = [spec.strip() for spec in row['Species'].split(';')]
    taxa = [taxon.strip() for taxon in row['Taxa Order'].split(';')]

    for taxon in taxa:
        for spec in species:
            years.append(year)
            taxa_orders.append(taxon)
            species_list.append(spec)

taxa_df = pd.DataFrame({
    'Year': years,
    'Taxa Order': taxa_orders,
    'Species': species_list
})

In [ ]:
taxa_df_counted = taxa_df.groupby(['Taxa Order', 'Species']).size().reset_index(name='Number of Studies')

# Create the Circular Treemap
fig = px.treemap(
    taxa_df_counted,
    path=['Taxa Order', 'Species'],
    values='Number of Studies',
    color='Taxa Order',  # You can choose a different column for color
    hover_data=['Number of Studies'],
    title='Circular Treemap of Taxa and Species by Number of Studies',
    #branch_values="total"  # Use total for circular layout
)

fig.update_layout(margin=dict(t=50, l=0, r=0, b=0))
fig.update_traces(textinfo='label+value')


# fig.write_image("circular_treemap.png")
# fig.write_image("circular_treemap.pdf")

fig.show()

In [ ]:
fig = px.sunburst(
    taxa_df_counted,
    path=['Taxa Order', 'Species'],
    values='Number of Studies',
    color='Taxa Order',
    hover_data=['Number of Studies'],
    # title='Sunburst Chart of Taxa and Species (Counts on Segments)'
)


fig.update_traces(textinfo='label+value')
fig.update_layout(margin=dict(t=50, l=0, r=0, b=0))

# Increase font size
fig.update_layout(
    width=800,
    height=800,
    font=dict(size=18), # applies to everything: titles, labels, legend, etc.
    xaxis=dict(title=dict(font=dict(size=23)), tickfont=dict(size=18)),
    yaxis=dict(title=dict(font=dict(size=23)), tickfont=dict(size=18))
)

# fig.write_image("sunburst_chart_static_counts.png")
# fig.write_image("sunburst_chart_static_counts.pdf")

fig.show()

In [ ]:
mutiple_to_single_species = {
    3: [
        "Rattus rattus",
        "Micromys minutus",
        "Rattus norvegicus",
        "Apodemus flavicollis",
        "Apodemus sylvaticus",
        "Mus musculus, Sorex araneus",
        "Arvicola amphibius",
        "Sorex minutus",
        "Muscardinus avellanarius",
        "Crocidura leucodon",
        "Apodemus agrarius",
        "Microtus agrestis",
        "Crocidura russula",
        "Glis glis",
        "Crocidura suaveolens",
        "Dryomys nitedula"
    ],
    9: [
        "Eptesicus fuscus",
        "Lasionycteris noctivagans",
        "Lasiurus borealis",
        "Lasiurus cinereus",
        "Myotis grisescens",
        "Myotis lucifugus",
        "Myotis sodalis",
        "Myotis septentrionalis",
        "Myotis leibii",
        "Nycticeius humeralis",
        "Perimyotis subflavus"
    ],
    11: [
        "Rhinopoma muscatellum",
        "Taphozous perforatus",
        "Pipistrellus kuhli",
        "Rhyneptesicus nasutus",
        "Eptesicus bottae",
        "Rousettus aegyptiacus",
        "Myotis emarginatus",
        "Asellia tridens"
    ],
    12: [
        "Pipistrellus pipistrellus",
        "Pipistrellus nathusii",
        "Pipistrellus pygmaeus",
        "Nyctalus noctula",
        "Barbastella barbastellus",
        "Myotis bechsteinii",
        "Myotis daubentonii",
        "Myotis myotis",
        "Myotis mystacinus",
        "Myotis nattereri",
        "Myotis brandtii",
        "Plecotus auritus",
        "Plecotus austriacus",
        "Nyctalus noctula",
        "Nyctalus leisleri",
        "Eptesicus serotinus",
        "Eptesicus nilssonii",
        "Vespertilio murinus"
    ],
    13: [
        "Barbastella pacifica",
        "Eptesicus japonensis",
        "Eptesicus nilssonii",
        "Hipposideros turpis",
        "Hypsugo alaschanicus",
        "Miniopterus fuliginosus",
        "Miniopterus fuscus",
        "Murina hilgendorfi",
        "Murina ryukyuana",
        "Murina ussuriensis",
        "Myotis bombinus",
        "Myotis frater",
        "Myotis gracilis",
        "Myotis ikonnikovi",
        "Myotis macrodactylus",
        "Myotis petax",
        "Myotis pruinosus",
        "Myotis rufoniger",
        "Myotis yanbarensis",
        "Nyctalus aviator",
        "Pipistrellus abramus",
        "Plecotus sacrimontis",
        "Rhinolophus cornutus cornutus",
        "Rhinolophus cornutus orii",
        "Rhinolophus ferrumequinum",
        "Rhinolophus pumilus",
        "Rhinolophus perditus",
        "Tadarida latouchei",
        "Vespertilio murinus",
        "Vespertilio sinensis"
    ],
    14: [

    ],
    15: [
        "Chimpanzees",
        "Mandrills",
        "Red-capped mangabeys",
        "Guenon species (mixed group)"
    ],
    16: [
        "Chimpanzees",
        "Mandrills",
        "Red-capped mangabeys",
        "Guenon species (mixed group)"
    ],
    17: [
        "Panthera pardus",
        "Neofelis nebulosa",
        "Panthera leo",
        "Panthera onca"
    ],
    24: [
        "Canis lupus",
        "Canis latrans"
    ],
}

In [ ]:
# import re

# def extract_scientific_names(text: str, par:bool=False) -> str:
#     """
#     Extracts the scientific names inside parentheses from a multiline string
#     and returns them as a comma-separated inline string.
#     """
#     if par:
#         names = re.findall(r"\((.*?)\)", text)
#     else:
#         names = [line.strip() for line in text.splitlines() if line.strip()]

#     names_quoted = [f'"{name}"' for name in names]

#     return ", ".join(names_quoted)


# # Example usage
# text = """
# Gray Wolf (Canis lupus)
# Coyote (Canis latrans)
# """

# extract_scientific_names(text, par=True)


In [ ]:
taxa_df_grouped_by_year = taxa_df.groupby(['Year', 'Taxa Order']).size().reset_index(name='Count')

# Create the stacked bar chart
fig = px.bar(
    taxa_df_grouped_by_year,
    x='Year',
    y='Count',
    color='Taxa Order',
    # title='Trend of Taxa Studies by Year',
    labels={'Count': 'Number of Studies', 'Taxa Order': 'Taxa Order'}
)

# Increase font size
fig.update_layout(
    font=dict(size=18), # applies to everything: titles, labels, legend, etc.
    xaxis=dict(title=dict(font=dict(size=23)), tickfont=dict(size=18)),
    yaxis=dict(title=dict(font=dict(size=23)), tickfont=dict(size=18))
)

# fig.write_html("taxa_studies_by_year_stacked_bar_loop.html")
# fig.write_image("taxa_studies_by_year_stacked_bar_loop.png")
# fig.write_image("taxa_studies_by_year_stacked_bar_loop.pdf")

fig.show()

## Geographical distribution

In [ ]:
df_geo_dist = df[['Region Broad', 'Region Specific']]
df_geo_dist['Region Specific'] = df_geo_dist['Region Specific'].fillna('N/A')

In [ ]:
region_broads = []
region_specifics = []

for index, row in df_geo_dist.iterrows():
    region_broad = [region.strip() for region in row['Region Broad'].split(';')]
    region_specific = [region.strip() for region in row['Region Specific'].split(';')]

    for broad in region_broad:
        for specific in region_specific:
            if specific != 'N/A':
                region_broads.append(broad)
                region_specifics.append(specific)

df_geo_dist = pd.DataFrame({
    'Region Broad': region_broads,
    'Region Specific': region_specifics
})

In [ ]:
# Count the number of studies per "Region Specific"
study_counts = df_geo_dist['Region Specific'].value_counts().reset_index()
study_counts.columns = ['Country', 'Number of Studies']

In [ ]:
# Create a dictionary to map country names to ISO alpha-3 codes
country_codes = {
    'China': 'CHN',
    'Japan': 'JPN',
    'UK': 'GBR',
    'Belarus': 'BLR',
    'Lithuania': 'LTU',
    'Madagascar': 'MDG',
    'United States of America': 'US',
    'Malaysia': 'MYS',
    'United Arab Emirates': 'ARE',
    'Germany': 'DEU',
    'South Korea': 'KOR',
    'N/A': 'N/A',
    'Cameroon': 'CMR',
    'USA': 'USA',
    # 'Republic of Korea': 'KOR',
    'Latvia': 'LVA',
    'Indonesia': 'IDN'
}

# Add an 'iso_alpha' column to the study_counts DataFrame
study_counts['iso_alpha'] = study_counts['Country'].map(country_codes).fillna('')

In [ ]:
fig_choropleth = px.choropleth(
    study_counts,
    locations='iso_alpha',
    locationmode='ISO-3',
    color='Number of Studies',
    hover_name='Country',
    color_continuous_scale=px.colors.sequential.Plasma,
    # title='Number of Studies per Country'
)

fig_choropleth.update_layout(margin=dict(l=0, r=0, t=20, b=20))
# fig_choropleth.write_image("world_heatmap_iso.png")
# fig_choropleth.write_image("world_heatmap_iso.pdf")

fig_choropleth.show()

In [ ]:
# study_counts = study_counts[~study_counts['Country'].isin(['Germany', 'UK'])]

In [ ]:
study_counts

In [ ]:
text_labels = [
    country if country not in ['Germany', 'UK', 'Latvia', 'Belarus', 'Japan', 'China', 'Indonesia'] else ""
    for country in study_counts['Country']
]

fig_bubble = px.scatter_geo(
    study_counts,
    locations='iso_alpha',
    locationmode='ISO-3',
    size='Number of Studies',
    hover_name='Country',
    text=text_labels, # 'iso_alpha'
    size_max=40,
    # title='Number of Studies per Country'
)

# Countries that need custom positions
custom_labels = {
    "UK": (53.0, -15.0),
    "Germany": (44.5, -12.0),
    "Latvia": (57.5, 26.0),
    "Belarus": (45, 30.5),
    "China": (45.0, 85.0),
    "Japan": (35.0, 155.0),
    "Indonesia": (-8.0, 118.0)
}

fig_bubble.update_traces(textposition="top center", textfont=dict(size=18))

# Add a new trace for custom labels
label_trace = go.Scattergeo(
    lon=[lon for _, lon in custom_labels.values()],
    lat=[lat for lat, _ in custom_labels.values()],
    text=list(custom_labels.keys()),
    mode="text",
    textfont=dict(size=18),
    showlegend=False
)

fig_bubble.add_trace(label_trace)

fig_bubble.update_layout(
    margin=dict(l=0, r=0, t=20, b=20),
    geo=dict(showland=True, landcolor="lightgray", showocean=False, oceancolor="lightblue"),
)

# fig_bubble.write_image("geospatial_bubble_chart_iso.png")
# fig_bubble.write_image("geospatial_bubble_chart_iso.pdf")

fig_bubble.show()

## Machine learning approaches


In [ ]:
# Dictionary to classify ML algorithms
ml_algorithm_types = {
    # Deep Learning models
    'CNN': "Deep Learning",
    'DCNN': "Deep Learning",
    'ResNet': "Deep Learning",
    'ResNet18': "Deep Learning",
    'ResNet50': "Deep Learning",
    'ResNet101': "Deep Learning",
    'ResNet152V2': "Deep Learning",
    'ResNet101V2': "Deep Learning",
    'ResNet50V2': "Deep Learning",
    'ResNet22': "Deep Learning",
    'WideResNet28-2': "Deep Learning",
    'DenseNet121': "Deep Learning",
    'DenseNet169': "Deep Learning",
    'DenseNet201': "Deep Learning",
    'VGG16': "Deep Learning",
    'VGGish': "Deep Learning",
    'MobileNetV1': "Deep Learning",
    'MobileNetV2': "Deep Learning",
    'InceptionV3': "Deep Learning",
    'InceptionResNetV2': "Deep Learning",
    'EfficientNet-B2': "Deep Learning",
    'Xception': "Deep Learning",
    'RNN': "Deep Learning",
    'LSTM': "Deep Learning",
    'BiLSTM': "Deep Learning",
    'BirdNET': "Deep Learning",

    # Traditional ML models
    'SVM': "Traditional ML",
    'RF': "Traditional ML",
    'KNN': "Traditional ML",
    'LR': "Traditional ML",
    'XGB': "Traditional ML",
    'LDA': "Traditional ML",
    'DT': "Traditional ML",
    'GNB': "Traditional ML",
    'MLP': "Traditional ML"
}

algorithm_family_mapping = {
    # CNN Family
    'CNN': 'Basic CNN',
    'DCNN': 'Basic CNN',
    'VGG16': 'VGG',
    'VGGish': 'VGG',
    'ResNet': 'ResNet',
    'ResNet18': 'ResNet',
    'ResNet22': 'ResNet',
    'ResNet50': 'ResNet',
    'ResNet50V2': 'ResNet',
    'ResNet101': 'ResNet',
    'ResNet101V2': 'ResNet',
    'ResNet152V2': 'ResNet',
    'WideResNet28-2': 'ResNet',
    'MobileNetV1': 'MobileNet',
    'MobileNetV2': 'MobileNet',
    'InceptionV3': 'Inception',
    'InceptionResNetV2': 'Inception',
    'DenseNet121': 'DenseNet',
    'DenseNet169': 'DenseNet',
    'DenseNet201': 'DenseNet',
    'EfficientNet-B2': 'Other - DL',
    'Xception': 'Other - DL',

    # RNN Family
    'RNN': 'Basic RNN',
    'LSTM': 'LSTM',
    'BiLSTM': 'LSTM',

    # Traditional ML
    'LR': 'Linear',
    'LDA': 'Linear',
    'DT': 'Tree-based',
    'RF': 'Tree-based',
    'XGB': 'Tree-based',
    'KNN': 'Other - TML',
    'SVM': 'Other - TML',
    'GNB': 'Other - TML',

    # Multi-Layer Networks
    'MLP': 'MLP',

    # Special Purpose Models
    'BirdNET': 'BirdNET'
}

In [ ]:
years = []
ml_categories = []
ml_algorithms = []
taxa_orders = []
species_list = []

for index, row in df.iterrows():
    year = row['Year']
    species = [spec.strip() for spec in row['Species'].split(';')]
    algorithms = [alg.strip() for alg in row['ML Algorithm Specific'].split(';')]
    taxa = [taxon.strip() for taxon in row['Taxa Order'].split(';')]

    for algorithm in algorithms:
        if algorithm in ml_algorithm_types:
            ml_category = ml_algorithm_types[algorithm]
        else:
            ml_category = "Unknown"
        for taxon in taxa:
            for spec in species:
                years.append(year)
                ml_categories.append(ml_category)
                ml_algorithms.append(algorithm)
                taxa_orders.append(taxon)
                species_list.append(spec)

ml_approaches_df = pd.DataFrame({
    'Year': years,
    'ML Algorithm Category': ml_categories,
    'ML Algorithm Specific': ml_algorithms,
    'Taxa Order': taxa_orders,
    'Species': species_list
})

In [ ]:
ml_approaches_df['Algorithm Family'] = ml_approaches_df['ML Algorithm Specific'].map(algorithm_family_mapping)

In [ ]:
# Group by Year and ML Algorithm Category and count occurrences
ml_approaches_df_grouped = ml_approaches_df.groupby(['Year', 'ML Algorithm Category']).size().reset_index(name='Count')

# Create the stacked bar chart
fig = px.bar(
    ml_approaches_df_grouped,
    x='Year',
    y='Count',
    color='ML Algorithm Category',
    # title='ML vs. DL Through Time',
    labels={'Count': 'Number of studies', 'ML Algorithm Category': 'ML Category'}
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=20), # applies to everything: titles, labels, legend, etc.
    xaxis=dict(title=dict(font=dict(size=23)), tickfont=dict(size=18)),
    yaxis=dict(title=dict(font=dict(size=23)), tickfont=dict(size=18))
)

# Save the plot
# fig.write_html("ml_dl_over_time_stacked_bar.html")
# fig.write_image("ml_dl_over_time_stacked_bar.png")
# fig.write_image("ml_dl_over_time_stacked_bar.pdf")

# Show the plot
fig.show()

In [ ]:
# Define the order of nodes for the Sankey diagram
ml_categories = sorted(ml_approaches_df['ML Algorithm Category'].dropna().unique())
taxa_orders_unique = sorted(ml_approaches_df['Taxa Order'].dropna().unique())
specific_algorithms_unique = sorted(ml_approaches_df['Algorithm Family'].dropna().unique())
# specific_algorithms_unique = sorted(ml_approaches_df['ML Algorithm Specific'].dropna().unique())
species_unique = sorted(ml_approaches_df['Species'].dropna().unique())

label_list = ml_categories + taxa_orders_unique + specific_algorithms_unique + species_unique
mapping_dict = {label: index for index, label in enumerate(label_list)}

source = []
target = []
value = []

for index, row in ml_approaches_df.iterrows():
    ml_cat = row['ML Algorithm Category']
    taxon = row['Taxa Order']
    algo = row['Algorithm Family']
    species = row['Species']

    if ml_cat in ml_categories and taxon in taxa_orders_unique:
        source.append(mapping_dict[ml_cat])
        target.append(mapping_dict[taxon])
        value.append(1)

    if taxon in taxa_orders_unique and algo in specific_algorithms_unique:
        source.append(mapping_dict[taxon])
        target.append(mapping_dict[algo])
        value.append(1)

    if algo in specific_algorithms_unique and species in species_unique:
        source.append(mapping_dict[algo])
        target.append(mapping_dict[species])
        value.append(1)

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=label_list,
        color="skyblue",
        # font=dict(size=18)
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=20),
)

# fig.update_layout(title_text="Flow: ML Category → Taxa Order → Specific Algorithm → Species", font_size=10)
# fig.write_image("sankey_ml_taxa_algo_species.png")
# fig.write_image("sankey_ml_taxa_algo_species.pdf")
fig.show()

In [ ]:
# Define color palette functions
def generate_colors_from_colormap(n, colormap_name):
    cmap = plt.get_cmap(colormap_name)
    if n == 1:
        return [mcolors.rgb2hex(cmap(0.5))]
    return [mcolors.rgb2hex(cmap(i / (n - 1))) for i in range(n)]

# Define the order of nodes for the Sankey diagram (Species removed)
ml_categories = sorted(ml_approaches_df['ML Algorithm Category'].dropna().unique())
taxa_orders_unique = sorted(ml_approaches_df['Taxa Order'].dropna().unique())
specific_algorithms_unique = sorted(ml_approaches_df['Algorithm Family'].dropna().unique())

label_list = ml_categories + taxa_orders_unique + specific_algorithms_unique
mapping_dict = {label: index for index, label in enumerate(label_list)}

# Create color mappings
ml_category_colors = dict(zip(
    ml_categories,
    generate_colors_from_colormap(len(ml_categories), 'viridis')
))
taxa_order_colors = dict(zip(
    taxa_orders_unique,
    generate_colors_from_colormap(len(taxa_orders_unique), 'coolwarm')
))

# Create node color list
node_colors = []

# ML category colors
for category in ml_categories:
    node_colors.append(ml_category_colors[category])

# Taxa order colors
for taxon in taxa_orders_unique:
    node_colors.append(taxa_order_colors[taxon])

# Algorithm family colors based on primary ML category
algo_colors = []
for algo in specific_algorithms_unique:
    algo_entries = ml_approaches_df[ml_approaches_df['Algorithm Family'] == algo]
    if len(algo_entries) > 0:
        primary_category = algo_entries['ML Algorithm Category'].mode()[0]
        algo_colors.append(ml_category_colors[primary_category])
    else:
        algo_colors.append("#CCCCCC")

node_colors.extend(algo_colors)

# Create Sankey links (Species removed)
source = []
target = []
value = []

for _, row in ml_approaches_df.iterrows():
    ml_cat = row['ML Algorithm Category']
    taxon = row['Taxa Order']
    algo = row['Algorithm Family']

    if pd.notna(ml_cat) and pd.notna(taxon):
        source.append(mapping_dict[ml_cat])
        target.append(mapping_dict[taxon])
        value.append(1)

    if pd.notna(taxon) and pd.notna(algo):
        source.append(mapping_dict[taxon])
        target.append(mapping_dict[algo])
        value.append(1)

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=label_list,
        color=node_colors
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    )
)])

fig.update_layout(
    font_size=20,
)

fig.show()


## Acoustic feature representations

In [ ]:
preprocessing_methods = []
data_representations = []
ml_categories = []
taxa_orders = []

for index, row in df.iterrows():
    methods = [prep.strip() for prep in row['Preprocessing Methods'].split(',')]
    representations = [repr.strip() for repr in row['Data Representation'].split(',')]
    categories = [cat.strip() for cat in row['ML Algorithm Category'].split(',')]
    taxa = [taxon.strip() for taxon in row['Taxa Order'].split(';')]

    for method in methods:
        for representation in representations:
            for category in categories:
                for taxon in taxa:
                    preprocessing_methods.append(method)
                    data_representations.append(representation)
                    ml_categories.append(category)
                    taxa_orders.append(taxon)

feature_representation_df = pd.DataFrame({
    'Preprocessing Methods': preprocessing_methods,
    'Data Representation': data_representations,
    'ML Algorithm Category': ml_categories,
    'Taxa Order': taxa_orders
})

In [ ]:
# Define node labels and their order
nodes = {
    'Preprocessing Methods': sorted(feature_representation_df['Preprocessing Methods'].dropna().unique()),
    'Data Representation': sorted(feature_representation_df['Data Representation'].dropna().unique()),
    'ML Algorithm Category': sorted(feature_representation_df['ML Algorithm Category'].dropna().unique()),
    # 'Taxa Order': sorted(feature_representation_df['Taxa Order'].dropna().unique())
}

all_nodes = []
for key in nodes:
    all_nodes.extend(nodes[key])

node_mapping = {node: i for i, node in enumerate(all_nodes)}

# Create links
links = []
cols = ['Preprocessing Methods', 'Data Representation', 'ML Algorithm Category', 'Taxa Order']
for i in range(len(cols) - 1):
    source_col = cols[i]
    target_col = cols[i + 1]
    temp_df = feature_representation_df.groupby([source_col, target_col]).size().reset_index(name='value')
    for _, row in temp_df.iterrows():
        source_node = row[source_col]
        target_node = row[target_col]
        if source_node in node_mapping and target_node in node_mapping:
            links.append({
                'source': node_mapping[source_node],
                'target': node_mapping[target_node],
                'value': row['value']
            })

# Prepare data for Sankey diagram
link_data = {
    'source': [link['source'] for link in links],
    'target': [link['target'] for link in links],
    'value': [link['value'] for link in links]
}

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_nodes,
        # color="#0E3C41"
        color="skyblue"
    ),
    link=dict(
        source=link_data['source'],
        target=link_data['target'],
        value=link_data['value'],
        # color="#869DA"
        # color="#B4C9CB"
    ))])

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=18),
    # family="Arial"
)

# fig.write_image("sankey_all_flows_loop.png")
# fig.write_image("sankey_all_flows_loop.pdf")
fig.show()

## Performance metrics and benchmarking

In [ ]:
def replace_values_with_column_names(data):
    perfs_df = pd.DataFrame(index=data.index)
    perfs_df['Year'] = data['Year']
    perfs_df['ML Algorithm Category'] = data['ML Algorithm Category']
    perfs_df['Taxa Order'] = data['Taxa Order']
    perfs_df['Accuracy'] = data['Accuracy']
    perfs_df['Precision'] = data['Precision']
    perfs_df['Recall'] = data['Recall']
    perfs_df['Specificity'] = data['Specificity']
    perfs_df['F1 Score'] = data['F1 Score']
    perfs_df['UAR'] = data['UAR']

    perf_metrics = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1 Score', 'UAR']

    metrics_by_row = {idx: [] for idx in perfs_df.index}

    for col in perfs_df.columns:
        new_series = pd.Series(index=perfs_df.index, dtype='object')

        for idx in perfs_df.index:
            if pd.notna(perfs_df.loc[idx, col]) and col in perf_metrics:
                metrics_by_row[idx].append(col)

    performance_metrics = pd.Series(index=perfs_df.index, dtype='object')
    for idx in perfs_df.index:
        if metrics_by_row[idx]:
            performance_metrics.loc[idx] = ';'.join(metrics_by_row[idx])
        else:
            performance_metrics.loc[idx] = ''

    perfs_df['Performance Metrics'] = performance_metrics

    return perfs_df


expanded_perfs_df = replace_values_with_column_names(df)

In [ ]:
def count_studies_with_both_metrics(data, year, metric1, metric2):
    """Counts studies in a given year that include both specified performance metrics."""
    return len(df[(data['Year'] == year) &
                   (data['Performance Metrics'].str.contains(metric1, case=False)) &
                   (data['Performance Metrics'].str.contains(metric2, case=False))])

# Metrics to check for co-occurrence
metric1 = 'Recall'
metric2 = 'Precision'
years_to_check = [2023, 2024]

results = {}

for year in years_to_check:
    count = count_studies_with_both_metrics(expanded_perfs_df, year, metric1, metric2)
    results[year] = {f'Both {metric1} and {metric2}': count}

# Print the results
for year, metric_counts in results.items():
    print(f"Studies in {year}:")
    for metric_combination, count in metric_counts.items():
        print(f"  - {metric_combination}: {count}")

In [ ]:
def count_metric_studies(data, year, metric):
    """Counts studies in a given year that include a specific performance metric."""
    return len(df[(data['Year'] == year) & (data['Performance Metrics'].str.contains(metric, case=False))])

# Performance metrics to check
metrics = ['Accuracy', 'F1 Score', 'Precision', 'Recall', 'Specificity', 'UAR']
years_to_check = [2020, 2021, 2022, 2023, 2024]

results = {}

for year in years_to_check:
    results[year] = {}
    for metric in metrics:
        count = count_metric_studies(expanded_perfs_df, year, metric)
        results[year][metric] = count

# Print the results
for year, metric_counts in results.items():
    print(f"Studies in {year}:")
    for metric, count in metric_counts.items():
        print(f"  - {metric}: {count}")

In [ ]:
years = []
ml_categories = []
taxa_orders = []
performance_metrics = []

for index, row in expanded_perfs_df.iterrows():
    year = row['Year']
    categories = [cat.strip() for cat in row['ML Algorithm Category'].split(',')]
    taxa = [taxon.strip() for taxon in row['Taxa Order'].split(';')]
    perfs = [perf.strip() for perf in row['Performance Metrics'].split(';')]

    for category in categories:
        for taxon in taxa:
            for perf in perfs:
                if perf == '':
                    perf = 'None'
                years.append(year)
                ml_categories.append(category)
                taxa_orders.append(taxon)
                performance_metrics.append(perf)

perfs_df = pd.DataFrame({
    'Year': years,
    'ML Algorithm Category': ml_categories,
    'Taxa Order': taxa_orders,
    'Performance Metrics': performance_metrics
})

In [ ]:
def create_sankey(data, path, title, filename):
    all_nodes = list(data[path[0]].unique())
    for col in path[1:]:
        all_nodes.extend(list(data[col].unique()))
    nodes = list(set(all_nodes))
    node_mapping = {node: i for i, node in enumerate(nodes)}

    links = []
    for i in range(len(path) - 1):
        source_col = path[i]
        target_col = path[i + 1]
        temp_df = data.groupby([source_col, target_col]).size().reset_index(name='value')
        for _, row in temp_df.iterrows():
            source = node_mapping[row[source_col]]
            target = node_mapping[row[target_col]]
            value = row['value']
            links.append({'source': source, 'target': target, 'value': value})

    link_data = {
        'source': [link['source'] for link in links],
        'target': [link['target'] for link in links],
        'value': [link['value'] for link in links]
    }

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=nodes,
            color="skyblue"
        ),
        link=dict(
            source=link_data['source'],
            target=link_data['target'],
            value=link_data['value']
        ))])

    fig.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(size=18),
        # family="Arial"
    )
    # fig.write_image(f"{filename}.png")
    # fig.write_image(f"{filename}.pdf")
    fig.show()

In [ ]:
# --- Sankey Diagram 1: Year → Performance Metric → Taxa ---
path = ['Year', 'Performance Metrics', 'Taxa Order']
title = "Sankey Diagram: Year → Performance Metric → Taxa"
filename = "sankey_year_perf_taxa"
create_sankey(perfs_df, path, title, filename)

In [ ]:
# --- Sankey Diagram 2: Year → Performance Metric → ML Category → Taxa ---
path = ['Performance Metrics', 'ML Algorithm Category']
title = "Sankey Diagram: Performance Metric → ML Category"
filename = "sankey_perf_ml"
create_sankey(perfs_df, path, title, filename)

In [ ]:
def plot_single_taxa_performance(data, taxa_name, ax=None, box_width=0.6):
    # Define performance metrics in the specified order
    performance_cols = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1 Score', 'UAR']

    # Filter data for the specific taxa
    taxa_data = data[data['Taxa Order'] == taxa_name].copy()

    if taxa_data.empty:
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, f'No data available\nfor {taxa_name}',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=12, style='italic')
        ax.set_title(f'{taxa_name} (n=0)')
        ax.set_ylim(50, 105)
        return ax

    # Prepare data for plotting
    plot_data = []
    sample_sizes = []

    for metric in performance_cols:
        if metric in taxa_data.columns:
            values = taxa_data[metric].dropna()
            if len(values) > 0:
                # Convert to 0-100 scale if needed
                if values.max() <= 1.0:
                    values = values * 100

                for value in values:
                    plot_data.append({'Metric': metric, 'Performance': value})
                sample_sizes.append(len(values))
            else:
                sample_sizes.append(0)
        else:
            sample_sizes.append(0)

    # Create DataFrame for plotting
    plot_df = pd.DataFrame(plot_data)

    # Create the plot
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    if not plot_df.empty:
        # Create box plot
        box_plot = sns.boxplot(
            data=plot_df,
            x='Metric',
            y='Performance',
            ax=ax,
            width=box_width,
            # color='white',
            order=performance_cols,
            linewidth=1,
            # boxprops=dict(edgecolor='black'),
            # whiskerprops=dict(color='black'),
            # capprops=dict(color='black'),
            # medianprops=dict(color='black'),
            color='#869DA0',
            boxprops=dict(edgecolor='#0E3C41'),
            whiskerprops=dict(color='#0E3C41'),
            capprops=dict(color='#0E3C41'),
            medianprops=dict(color='#0E3C41'),  # white median for contrast
        )

        # Add sample size annotations
        for i, (metric, n) in enumerate(zip(performance_cols, sample_sizes)):
            if n > 0:
                # Get the maximum value for this metric to position the annotation
                metric_values = plot_df[plot_df['Metric'] == metric]['Performance']
                if not metric_values.empty:
                    y_pos = metric_values.max() + 2
                    ax.text(i, y_pos, f'n={n}',
                           ha='center', va='bottom', fontsize=9,
                           bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

    # Customize the plot
    ax.set_ylim(50, 105)
    ax.set_ylabel('Performance (%)', fontsize=13)
    ax.set_xlabel('Performance Metrics', fontsize=13)
    ax.set_title(f'{taxa_name}', fontsize=13)

    # Rotate x-axis labels for better readability
    ax.tick_params(axis='x', rotation=90, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)

    # Add grid for better readability
    ax.grid(True, alpha=0.3, axis='y')

    return ax

def plot_all_taxa_performance_grid(data, figsize=(15, 10), box_width=0.6):
    taxa_list = data['Taxa Order'].unique().tolist()

    while len(taxa_list) < 6:
        taxa_list.append(None)

    # Create 2x3 subplot grid
    fig, axes = plt.subplots(2, 3, figsize=figsize, sharex=True, sharey=True)
    axes = axes.flatten()

    # Plot each taxa
    for i, taxa in enumerate(taxa_list[:6]):
        if taxa is not None:
            plot_single_taxa_performance(data, taxa, ax=axes[i], box_width=box_width)
        else:
            # Empty subplot for padding
            axes[i].axis('off')
            axes[i].text(0.5, 0.5, 'No data', ha='center', va='center',
                        transform=axes[i].transAxes, fontsize=18, style='italic')

    # Adjust layout
    plt.tight_layout()

    return fig, axes

In [ ]:
fig, axes = plot_all_taxa_performance_grid(df.drop(14), figsize=(12, 7), box_width=0.4)
plt.suptitle('', fontsize=16, fontweight='bold', y=1.02)

In [ ]:
fig.savefig("performance_across_taxa.png", dpi=700, bbox_inches="tight")

In [ ]:
def print_performance_statistics_by_taxa(data):
    performance_cols = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1 Score', 'UAR']

    print("Performance Statistics by Taxonomic Group")
    print("=" * 80)

    for taxa in sorted(data['Taxa Order'].unique()):
        print(f"\n{taxa.upper()}:")
        print("-" * 40)

        taxa_data = data[data['Taxa Order'] == taxa]

        for metric in performance_cols:
            if metric in taxa_data.columns:
                values = taxa_data[metric].dropna()

                if len(values) > 0:
                    # Convert to 0-100 scale if needed
                    if values.max() <= 1.0:
                        values = values * 100

                    stats = {
                        'n': len(values),
                        'min': values.min(),
                        'max': values.max(),
                        'mean': values.mean(),
                        'median': values.median(),
                        'std': values.std()
                    }

                    print(f"  {metric:12} (n={stats['n']:2}): "
                          f"min={stats['min']:5.1f}%, max={stats['max']:5.1f}%, "
                          f"mean={stats['mean']:5.1f}±{stats['std']:4.1f}%, "
                          f"median={stats['median']:5.1f}%")
                else:
                    print(f"  {metric:12}: No data available")

In [ ]:
df[df['Taxa Order'].str.upper() == 'ARTIODACTYLS'][['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1 Score', 'UAR']]

In [ ]:
print_performance_statistics_by_taxa(df.drop(14))

In [ ]:
def get_highest_metric_per_study(data):
    performance_cols = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1 Score', 'UAR']

    # Select only performance columns that exist in the dataframe
    available_cols = [col for col in performance_cols if col in data.columns]

    if not available_cols:
        return pd.Series(dtype=float)

    # Get performance data and convert to 0-100 scale if needed
    perf_data = data[available_cols].copy()

    # Convert to 0-100 scale if values are in 0-1 range
    for col in available_cols:
        if perf_data[col].max() <= 1.0:
            perf_data[col] = perf_data[col] * 100

    # Get the maximum value across all metrics for each study
    highest_values = perf_data.max(axis=1)

    return highest_values

def categorize_performance(value):
    if pd.isna(value):
        return 'No data'
    elif value >= 95:
        return 'Very High (≥95%)'
    elif value >= 85:
        return 'High (85-94%)'
    elif value >= 70:
        return 'Moderate (70-84%)'
    else:
        return 'Low (<70%)'

def plot_performance_threshold_colorful_pie(data, figsize=(8, 8)):
    highest_metrics = get_highest_metric_per_study(data)

    if highest_metrics.empty:
        print("No performance data available for pie chart.")
        return None, None

    categories = highest_metrics.apply(categorize_performance)
    category_counts = categories.value_counts()

    colors = {
        'Very High (≥95%)': '#2E8B57',  # Sea Green
        'High (85-94%)': '#90EE90',     # Light Green
        'Moderate (70-84%)': '#FFD700', # Gold
        'Low (<70%)': '#FF6347',        # Tomato
        'No data': '#D3D3D3'            # Light Gray
    }

    fig, ax = plt.subplots(figsize=figsize)

    # Get colors for the categories present in data
    pie_colors = [colors.get(cat, '#D3D3D3') for cat in category_counts.index]

    # Create pie chart (remove labels)
    wedges, _, autotexts = ax.pie(
        category_counts.values,
        labels=None,
        colors=pie_colors,
        autopct='%1.1f%%',
        startangle=90,
        explode=[0.02] * len(category_counts),
        wedgeprops=dict(edgecolor='black', linewidth=1),
        textprops={'fontsize': 12}
    )

    # Make percentage text bold
    for autotext in autotexts:
        autotext.set_fontweight('bold')
        autotext.set_color('white')

    # Add legend instead of texts
    ax.legend(
        wedges,
        category_counts.index,
        # title="Performance Categories",
        loc="upper left",
        bbox_to_anchor=(1, 0.5),
        fontsize=10
    )

    plt.tight_layout()

    return fig, ax

In [ ]:
fig, ax = plot_performance_threshold_colorful_pie(df.drop(14))


fig.savefig(
    "performance_threshold_pie.png",
    dpi=700,
    bbox_inches="tight"
)

## Methodological quality assessment

In [ ]:
25+62.5

## Utility fonctions

In [ ]:
def replace_commas(data):
    for col in data.select_dtypes(include=['object']).columns:
        data[col] = data[col].astype(str).str.replace(',', ';')
    return data

df_to_save = expanded_perfs_df[["Year","Performance Metrics", "ML Algorithm Category"]]
df_to_save = replace_commas(df_to_save)

output_file = "taxa_studies_data.txt"
df_to_save.to_csv(output_file, sep=',', index=False)

print(f"Data successfully written to {output_file}")

In [ ]:
def analyze_year_taxa(data, year, taxa):
    df_year = data[data['Year'] == year].copy()

    def count_metrics(metrics_str):
        if isinstance(metrics_str, str):
            return len(metrics_str.split(';'))
        return 0

    df_year['Metric_Count'] = df_year['Performance Metrics'].apply(count_metrics)

    df_taxa = df_year[df_year['Taxa Order'].str.contains(taxa, case=False)]
    metric_counts = df_taxa['Metric_Count'].tolist()
    num_studies = len(df_taxa)

    all_metrics_within_4_5 = all(4 <= count <= 5 for count in metric_counts)

    return {
        'num_studies': num_studies,
        'metric_counts': metric_counts,
        'all_metrics_within_4_5': all_metrics_within_4_5
    }

In [ ]:
# Find all unique taxa
all_taxa_lists = df['Taxa Order'].apply(lambda x: [t.strip() for t in x.split(';')] if isinstance(x, str) else [x])
unique_taxa = set()
for taxa_list in all_taxa_lists:
    unique_taxa.update(taxa_list)

In [ ]:
# Analyze for a specific year
year_to_analyze = 2023

for taxon in sorted(list(unique_taxa)): # Iterate through the set of unique taxa
    analysis_results = analyze_year_taxa(expanded_perfs_df, year_to_analyze, taxon)
    print(f"\nAnalysis for {taxon} in {year_to_analyze}:")
    print(f"  Number of studies: {analysis_results['num_studies']}")
    print(f"  Metric counts per study: {analysis_results['metric_counts']}")
    print(f"  All studies reported 4-5 metrics: {analysis_results['all_metrics_within_4_5']}")

In [ ]:
# Analyze for a specific year
year_to_analyze = 2024

for taxon in sorted(list(unique_taxa)): # Iterate through the set of unique taxa
    analysis_results = analyze_year_taxa(expanded_perfs_df, year_to_analyze, taxon)
    print(f"\nAnalysis for {taxon} in {year_to_analyze}:")
    print(f"  Number of studies: {analysis_results['num_studies']}")
    print(f"  Metric counts per study: {analysis_results['metric_counts']}")
    print(f"  All studies reported 4-5 metrics: {analysis_results['all_metrics_within_4_5']}")

In [ ]:

df_2024 = expanded_perfs_df[expanded_perfs_df['Year'] == 2024].copy()

# Function to count the number of metrics in a string
def count_metrics(metrics_str):
    if isinstance(metrics_str, str):
        return len(metrics_str.split(';'))
    return 0

# Apply the function to count metrics per study
df_2024['Metric_Count'] = df_2024['Performance Metrics'].apply(count_metrics)

# Analyze Artiodactyls in 2024
artiodactyls_2024 = df_2024[df_2024['Taxa Order'].str.contains('Artiodactyls', case=False)]
artiodactyls_metric_counts = artiodactyls_2024['Metric_Count']
num_artiodactyl_studies_2024 = len(artiodactyls_2024)

print("\nPerformance metric counts for Artiodactyls in 2024:")
print(artiodactyls_metric_counts)
print(f"Number of Artiodactyl studies in 2024: {num_artiodactyl_studies_2024}")

# Analyze Primates in 2024
primates_2024 = df_2024[df_2024['Taxa Order'].str.contains('Primates', case=False)]
primates_metric_counts = primates_2024['Metric_Count']
num_primate_studies_2024 = len(primates_2024)

print("\nPerformance metric counts for Primates in 2024:")
print(primates_metric_counts)
print(f"Number of Primate studies in 2024: {num_primate_studies_2024}")

# Count the total number of studies in 2024
num_studies_2024 = len(df_2024)
print(f"\nTotal number of studies in 2024: {num_studies_2024}")

In [ ]:
expanded_perfs_df[expanded_perfs_df['ML Algorithm Category'].str.contains('Deep', case=False)].sort_values(by='Year').reset_index(drop=True)

In [ ]:
expanded_perfs_df[expanded_perfs_df['Taxa Order'].str.contains('Tree', case=False)].sort_values(by='Year').reset_index(drop=True)

# Quality assessment

In [ ]:
url = ROOT / 'data' / 'processed' / 'quality_assessment.csv'


In [ ]:
choice_to_score = {
    "0: Not reported/insufficient": 0,
    "1: Partially justified": 1,
    "2: Fully justified with rationale": 2,

    "0: Not addressed": 0,
    "1: Acknowledged but not handled": 1,
    "2: Properly addressed with methods": 2,

    "0: Not described": 0,
    "1: Partially described": 1,
    "2: Fully described": 2,

    "0: No justification": 0,
    "1: Partial justification": 1,
    "2: Comprehensive justification": 2,

    "1: Basic justification": 1,
    "2: Full comparison": 2,

    "0: No validation": 0,
    "1: Basic validation": 1,
    "2: Rigorous validation": 2,

    "0: Incomplete": 0,
    "1: Basic description": 1,
    "2: Detailed description": 2,

    "0: Not available": 0,
    "1: Partially available": 1,
    "2: Fully available with documentation": 2,

    "1: Available upon request": 1,
    "2: Publicly available with documentation": 2,

    "0: Not documented": 0,
    "1: Partially documented": 1,
    "2: Fully documented": 2,

    "0: No comparison": 0,
    "1: Limited comparison": 1,
    "2: Comprehensive comparison": 2,

    "0: Basic metrics only": 0,
    "1: Multiple metrics": 1,
    "2: Full evaluation with analysis": 2,

    "0: Not discussed": 0,
    "1: Briefly mentioned": 1,
    "2: Fully analyzed": 2,

    "1: Briefly mentioned": 1,
    "2: Detailed discussion": 2
}

In [ ]:
words_to_remove = [
    "Paper information",
    "Data quality",
    "Method practices",
    "Transparency",
    "Research completeness",
    "Overall evaluation"
]

In [ ]:
qual_assessment_data = pd.read_csv(url)
qual_assessment_data = qual_assessment_data.loc[:, ~qual_assessment_data.columns.str.contains("Rationale", case=False)]

In [ ]:
def clean_column(col_name):
    for word in words_to_remove:
        if col_name.startswith(word) and col_name != "Data quality control":
            col_name = col_name.replace(word, "").strip()
    return col_name

In [ ]:
papers_scores_df = qual_assessment_data.replace(choice_to_score)

In [ ]:
papers_scores_df.columns = [clean_column(col) for col in papers_scores_df.columns]

idx = papers_scores_df.columns.get_loc("Future directions") + 1
papers_scores = papers_scores_df.iloc[:, :idx]

non_score_cols = ['id', 'Title', 'Authors', 'Year', 'DOI/Link']
score_columns = ['Data volume', 'Data balance', 'Data quality control', 'Model selection', 'Model validation', 'Method description', 'Code availability', 'Data availability', 'Technical specifications', 'Comparative analysis', 'Performance evaluation', 'Ecological relevance', 'Future directions']

In [ ]:
papers_scores_df.drop(['Title', 'DOI/Link', 'Authors'], axis=1, inplace=True)

In [ ]:
unnamed_cols = [col for col in papers_scores_df.columns if "Unnamed" in col]
papers_scores_df.drop(columns=unnamed_cols, inplace=True, axis=1)

In [ ]:
papers_scores_df.rename(
    columns={
        'Score': 'Score Data quality',
        'Score.1': 'Score Method practices',
        'Score.2': 'Score Transparency',
        'Score.3': 'Score Research completeness'
        },
    inplace=True
)

In [ ]:
overall_score_cols = [
    'id',  'Year',
    'Score Data quality',
    'Score Method practices',
    'Score Transparency',
    'Score Research completeness',
    'Total Score'
]
papers_group_score_df = papers_scores_df[overall_score_cols]

In [ ]:
for col in papers_group_score_df.columns[2:]:
    papers_group_score_df.loc[:, col] = papers_group_score_df[col].astype(float)

In [ ]:
papers_group_score_df.loc[:, 'Score Data quality'] = papers_group_score_df['Score Data quality'] / 6
papers_group_score_df.loc[:, 'Score Method practices'] = papers_group_score_df['Score Method practices'] / 4
papers_group_score_df.loc[:, 'Score Transparency'] = papers_group_score_df['Score Transparency'] / 8
papers_group_score_df.loc[:, 'Score Research completeness'] = papers_group_score_df['Score Research completeness'] / 8
papers_group_score_df.loc[:, 'Total Score'] = papers_group_score_df['Total Score'] / 26

In [ ]:
papers_group_score_df.head()

## Overall quality assessment results

### Temporal Trend of Quality Scores

In [ ]:
long_df = papers_group_score_df.melt(
    id_vars=['Year'],
    value_vars=[
        'Score Data quality',
        'Score Method practices',
        'Score Transparency',
        'Score Research completeness',
        'Total Score',
    ],
    var_name='Score Category',
    value_name='Score'
)

# Group by year and score category to get average scores
avg_scores_by_year = long_df.groupby(['Year', 'Score Category'])['Score'].mean().reset_index()

fig1 = px.line(
    avg_scores_by_year,
    x='Year',
    y='Score',
    color='Score Category',
    markers=True,
    # title='Temporal Trend of Quality Scores'
)
fig1.update_xaxes(type='category')
fig1.show()


### Histogram of Total Quality Scores

In [ ]:
fig2 = px.histogram(
    papers_group_score_df,
    x='Total Score',
    nbins=10,
    title='Histogram of Total Quality Scores'
)

# Add reference lines
fig2.add_vline(x=0.8, line_dash="dash", line_color="green", annotation_text="Excellent > 0.8", annotation_position="top right")
fig2.add_vline(x=0.7, line_dash="dash", line_color="orange", annotation_text="Good > 0.7", annotation_position="top right")
fig2.add_vline(x=0.6, line_dash="dash", line_color="red", annotation_text="Fair > 0.6", annotation_position="top right")

fig2.show()


### Boxplot of Quality Scores by Category

In [ ]:
# Melt again to include total score
box_df = papers_group_score_df.melt(
    id_vars=['id'],
    value_vars=[
        'Score Data quality',
        'Score Method practices',
        'Score Transparency',
        'Score Research completeness',
        'Total Score'
    ],
    var_name='Score Category',
    value_name='Score'
)

fig3 = px.box(
    box_df,
    x='Score Category',
    y='Score',
    # title='Boxplot of Quality Scores by Category'
)
fig3.show()


In [ ]:
# Melt again to include total score
box_df = papers_group_score_df.melt(
    id_vars=['id'],
    value_vars=[
        'Score Data quality',
        'Score Method practices',
        'Score Transparency',
        'Score Research completeness',
        'Total Score'
    ],
    var_name='Score Category',
    value_name='Score'
)

fig3 = px.box(
    box_df,
    x='Score Category',
    y='Score',
    # title='Boxplot of Quality Scores by Category'
)

# Update layout for white background and reduce box width
fig3.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    title=None,
    xaxis=dict(
        showgrid=True,
        gridcolor='lightgray',
        linecolor='black',
        linewidth=1
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgray',
        linecolor='black',
        linewidth=1
    ),
    width=900,  # Reduce overall figure width to bring boxes closer
    margin=dict(l=50, r=50, t=50, b=100)  # Adjust margins
)

# Reduce box width and change color to gray
fig3.update_traces(
    width=0.4,
    marker=dict(color='#0E3C41'),
    line=dict(color='#0E3C41')
)

fig3.show()

In [ ]:
detailed_papers_scores = papers_scores_df.drop(columns=overall_score_cols)
category_columns = detailed_papers_scores.columns

In [ ]:
# Define color mapping
cmap = plt.cm.colors.ListedColormap(['red', 'yellow', 'green'])
bounds = [-0.5, 0.5, 1.5, 2.5]
norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)

plt.figure(figsize=(10, 8))
sns.heatmap(detailed_papers_scores, annot=True, cmap=cmap, norm=norm, cbar=False, fmt=".0f")

plt.xticks(np.arange(len(category_columns)) + 0.5, category_columns, rotation=90, ha='right')
plt.yticks(np.arange(len(detailed_papers_scores)) + 0.5, detailed_papers_scores.index + 1)
plt.xlabel('Category')
plt.ylabel('Paper ID')
# plt.title('Scores Heatmap by Category for Each Paper')
plt.tight_layout()
plt.savefig("scores_heatmap_by_category.png", dpi=700, bbox_inches="tight")

In [ ]:
# df_to_save = df[[
#     'id',
#     'ML Algorithm Category',
#     'ML Algorithm Specific',
#     'Taxa Order',
#     'Species',
#     'Accuracy',
#     'Precision',
#     'Recall',
#     'Specificity',
#     'F1 Score',
#     'UAR',
# ]]

# output_file = "group_scores.txt"

# df_to_save.to_csv(output_file, sep=',', index=False)

# print(f"Data successfully written to {output_file}")

In [ ]:
df.columns

In [ ]:
df['Evaluation Method'].value_counts()

# Gap analysis and research priorities

In [ ]:
df

In [ ]:
# Heatmap of IUCN conservation list vs number of studies


## World map showing the location of studies

In [ ]:
# Use Region Specific if available, else Region Broad
def get_locations(row):
    if pd.notna(row['Region Specific']) and row['Region Specific'].strip() != '':
        return [x.strip() for x in str(row['Region Specific']).split(';')]
    elif pd.notna(row['Region Broad']):
        return [x.strip() for x in str(row['Region Broad']).split(';')]
    else:
        return []

df['Locations'] = df.apply(get_locations, axis=1)
data_expanded = df.explode('Locations')

# Group by country and count studies
loc_counts = data_expanded['Locations'].value_counts().reset_index()
loc_counts.columns = ['Country', 'Study Count']

# Plot world map
fig = px.choropleth(
    loc_counts,
    locations='Country',
    locationmode='country names',
    color='Study Count',
    hover_name='Country',
    color_continuous_scale='Viridis',
    title='Global Distribution of Bioacoustics Studies'
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

## Box plot: Dataset size distribution across Taxa Order to highlight imbalanced representation.

In [ ]:
# Numeric conversion
for col in ['Dataset Size Train', 'Dataset Size Validation', 'Dataset Size Testing']:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

# Compute total dataset size
df['Total Dataset Size'] = df[['Dataset Size Train', 'Dataset Size Validation', 'Dataset Size Testing']].sum(axis=1, skipna=True)

fig = px.box(df, x='Taxa Order', y='Total Dataset Size',
             title='Dataset Size Distribution Across Taxa Order',
             labels={'Total Dataset Size': 'Total Dataset Size', 'Taxa Order': 'Taxa Order'})

fig.update_layout(xaxis_title='Taxa Order', yaxis_title='Total Dataset Size', showlegend=False)
fig.show()

In [ ]:
selected_columns_for_transparency_gaps = df[[
    'Data Availability',
    'DOI/Link',
    'Recommendations for Future Research'
]]

# Save to a comma-separated text file
selected_columns_for_transparency_gaps.to_csv('transparency_gaps_analysis_data.txt', index=False, sep=',')

print("The selected columns have been exported to 'transparency_gaps_analysis_data.txt'")

## Scatter plot with trend line: Dataset Size (Total Duration Hours) vs. Year


In [ ]:
# Numeric conversion
for col in ['Dataset Size Train', 'Dataset Size Validation', 'Dataset Size Testing', 'Total Duration Hours']:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

df = df[pd.to_numeric(df['Year'], errors='coerce').notnull()]
df['Year'] = df['Year'].astype(int)

# Plot
fig = px.scatter(
    df, x='Year', y='Total Duration Hours',
    color='Data Availability',
    trendline='ols',
    title='Dataset Size (Total Duration Hours) vs. Year by Data Availability',
    labels={'Total Duration Hours': 'Total Duration (Hours)', 'Year': 'Year'}
)

# Force integer ticks for years
years = np.sort(df['Year'].unique())
fig.update_xaxes(
    tickmode='array',
    tickvals=years,
    ticktext=[str(int(y)) for y in years]
)

fig.update_layout(xaxis_title='Year', yaxis_title='Total Duration (Hours)', legend_title='Data Availability')
fig.show()

## Stacked bar chart: Data Availability categories

In [ ]:
df = df[pd.to_numeric(df['Year'], errors='coerce').notnull()]
df['Year'] = df['Year'].astype(int)
df['Data Availability'] = df['Data Availability'].str.strip()

# Count occurrences
bar_df = df.groupby(['Year', 'Data Availability']).size().reset_index(name='Count')

# Plot stacked bar chart by year
fig = px.bar(
    bar_df,
    x='Year',
    y='Count',
    color='Data Availability',
    title='Data Availability of Datasets Across Years (Stacked Bar Chart)',
    labels={'Count': 'Number of Studies', 'Year': 'Year'},
)

fig.update_layout(barmode='stack', xaxis={'type': 'category'})
fig.show()

In [ ]:
bar_df2 = df.groupby(['Taxa Order', 'Data Availability']).size().reset_index(name='Count')

fig2 = px.bar(
    bar_df2,
    x='Taxa Order',
    y='Count',
    color='Data Availability',
    title='Data Availability of Datasets Across Taxa Order (Stacked Bar Chart)',
    labels={'Count': 'Number of Studies', 'Taxa Order': 'Taxonomic Order'},
)

fig2.update_layout(barmode='stack', xaxis={'categoryorder': 'total descending'})
fig2.show()

## Grouped bar chart: "Evaluation Method" categories vs. count

In [ ]:
# Strip whitespace and count occurrences
df['Evaluation Method'] = df['Evaluation Method'].str.strip()
eval_counts = df['Evaluation Method'].value_counts().reset_index()
eval_counts.columns = ['Evaluation Method', 'Count']

# Plot grouped bar chart
fig = px.bar(
    eval_counts,
    x='Evaluation Method',
    y='Count',
    text='Count',
    title='Number of Studies by Evaluation Method',
    labels={'Evaluation Method': 'Evaluation Method', 'Count': 'Number of Studies'},
)

fig.update_layout(barmode='group', xaxis={'categoryorder': 'total descending'})
fig.show()

## Read all species in specific Taxa

In [ ]:
def get_order_subset(taxa_order):
    return df[df['Taxa Order'].str.contains(taxa_order)][["id", "Species"]]

def extract_unique_species(order_species):
    all_species = (
        order_species["Species"]
        .dropna()
        .apply(lambda x: [s.strip() for s in str(x).split(",")])
        .sum()
    )

    return sorted(set(all_species))

In [ ]:
primate_species = get_order_subset("Primates")
primate_species.loc[14, 'Species'] = "Pan troglodytes"
primate_species.loc[15, 'Species'] = "Pan troglodytes, Mandrillus sphinx, Cercocebus torquatus, Cercopithecus spp.)"
primate_species.loc[16, 'Species'] = "Pan troglodytes, Mandrillus sphinx, Cercocebus torquatus, Cercopithecus spp.)"
unique_primate_species = extract_unique_species(primate_species)
print(unique_primate_species)

In [ ]:
chiroptera_species = get_order_subset("Chiroptera")
chiroptera_species.loc[9, 'Species'] = "Eptesicus fuscus, Lasionycteris noctivagans, Lasiurus borealis, Lasiurus cinereus, Myotis grisescens, Myotis lucifugus, Myotis sodalis, Myotis septentrionalis, Myotis leibii, Nycticeius humeralis, Perimyotis subflavus"
chiroptera_species.loc[11, 'Species'] = "Rhinopoma muscatellum, Taphozous perforatus, Pipistrellus kuhli, Rhyneptesicus nasutus, Eptesicus bottae, Rousettus aegyptiacus, Myotis emarginatus, Asellia tridens"
chiroptera_species.loc[12, 'Species'] = "Pipistrellus pipistrellus, Pipistrellus nathusii, Pipistrellus pygmaeus, Nyctalus noctula, Barbastella barbastellus, Myotis bechsteinii, Myotis daubentonii, Myotis myotis, Myotis mystacinus, Myotis nattereri, Myotis brandtii, Plecotus auritus, Plecotus austriacus, Nyctalus noctula, Nyctalus leisleri, Eptesicus serotinus, Eptesicus nilssonii, Vespertilio murinus"
chiroptera_species.loc[13, 'Species'] = "Barbastella pacifica, Eptesicus japonensis, Eptesicus nilssonii, Hipposideros turpis, Hypsugo alaschanicus, Miniopterus fuliginosus, Miniopterus fuscus, Murina hilgendorfi, Murina ryukyuana, Murina ussuriensis, Myotis bombinus, Myotis frater, Myotis gracilis, Myotis ikonnikovi, Myotis macrodactylus, Myotis petax, Myotis pruinosus, Myotis rufoniger, Myotis yanbarensis, Nyctalus aviator, Pipistrellus abramus, Plecotus sacrimontis, Rhinolophus cornutus cornutus, Rhinolophus cornutus orii, Rhinolophus ferrumequinum, Rhinolophus pumilus, Rhinolophus perditus, Tadarida latouchei, Vespertilio murinus, Vespertilio sinensis"
unique_chiroptera_species = extract_unique_species(chiroptera_species)
print(unique_chiroptera_species)

In [ ]:
carnivora_species = get_order_subset("Carnivora")
carnivora_species.loc[14, 'Species'] = "Canis familiaris"
carnivora_species.loc[17, 'Species'] = "Panthera pardus, Neofelis nebulosa, Panthera leo, Panthera onca"
carnivora_species.loc[24, 'Species'] = "Canis lupus, Canis latrans"
unique_carnivora_species = extract_unique_species(carnivora_species)
print(unique_carnivora_species)

In [ ]:
artiodactyls_species = get_order_subset("Artiodactyls")
unique_artiodactyls_species = extract_unique_species(artiodactyls_species)
unique_artiodactyls_species

In [ ]:
rodentia_species = get_order_subset("Rodentia")
rodentia_species.loc[2, "Species"] = "Rattus rattus, Micromysminutus, Rattus norvegicus, Apodemus flavicollis, Apodemus sylvaticus, Mus musculus, Sorex araneus, Arvicola amphibius, Sorex minutus, Muscardinus avellanarius, Crocidura leucodon, Apodemus agrarius, Microtus agrestis, Crocidura russula, Glis glis, Crocidura suaveolens, Neomys fodiens, Dryomys nitedula"
unique_rodentia_species = extract_unique_species(rodentia_species)
print(unique_rodentia_species)

In [ ]:
treeshrew_species = get_order_subset("Treeshrew")
treeshrew_species.loc[14, "Species"] = "Tupaia belangeri"
unique_treeshrew_species = extract_unique_species(treeshrew_species)
unique_treeshrew_species

In [ ]:
proboscideans_species = get_order_subset("Proboscideans")
unique_proboscideans_species = extract_unique_species(proboscideans_species)
unique_proboscideans_species

In [ ]:
# --- Species data per order ---
species_by_order = {
    "Primates": ['Cercocebus torquatus', 'Cercopithecus spp.', 'Hylobates albibarbis', 'Hylobates funereus',
                 'Hylobates muelleri', 'Indri indri', 'Mandrillus sphinx', 'Nomascus hainanus',
                 'Pan troglodytes', 'Varecia variegata'],
    "Chiroptera": ['Asellia tridens', 'Barbastella barbastellus', 'Barbastella pacifica', 'Eptesicus bottae',
                    'Eptesicus fuscus', 'Eptesicus japonensis', 'Eptesicus nilssonii', 'Eptesicus serotinus',
                    'Hipposideros turpis', 'Hypsugo alaschanicus', 'Lasionycteris noctivagans', 'Lasiurus borealis',
                    'Lasiurus cinereus', 'Miniopterus fuliginosus', 'Miniopterus fuscus', 'Murina hilgendorfi',
                    'Murina ryukyuana', 'Murina ussuriensis', 'Myotis bechsteinii', 'Myotis bombinus',
                    'Myotis brandtii', 'Myotis daubentonii', 'Myotis emarginatus', 'Myotis frater',
                    'Myotis gracilis', 'Myotis grisescens', 'Myotis ikonnikovi', 'Myotis leibii',
                    'Myotis lucifugus', 'Myotis macrodactylus', 'Myotis myotis', 'Myotis mystacinus',
                    'Myotis nattereri', 'Myotis petax', 'Myotis pruinosus', 'Myotis rufoniger',
                    'Myotis septentrionalis', 'Myotis sodalis', 'Myotis yanbarensis', 'Nyctalus aviator',
                    'Nyctalus leisleri', 'Nyctalus noctula', 'Nycticeius humeralis', 'Perimyotis subflavus',
                    'Pipistrellus abramus', 'Pipistrellus kuhli', 'Pipistrellus nathusii', 'Pipistrellus pipistrellus',
                    'Pipistrellus pygmaeus', 'Plecotus auritus', 'Plecotus austriacus', 'Plecotus sacrimontis',
                    'Rhinolophus cornutus cornutus', 'Rhinolophus cornutus orii', 'Rhinolophus ferrumequinum',
                    'Rhinolophus perditus', 'Rhinolophus pumilus', 'Rhinopoma muscatellum', 'Rhyneptesicus nasutus',
                    'Rousettus aegyptiacus', 'Tadarida latouchei', 'Taphozous perforatus', 'Vespertilio murinus',
                    'Vespertilio sinensis'],
    "Carnivora": ['Canis familiaris', 'Canis latrans', 'Canis lupus', 'Neofelis nebulosa',
                  'Panthera leo', 'Panthera onca', 'Panthera pardus'],
    "Artiodactyla": ['Cervus elaphus', 'Cervus nippon', 'Sus scrofa domestica'],
    "Rodentia": ['Apodemus agrarius', 'Apodemus flavicollis', 'Apodemus sylvaticus', 'Arvicola amphibius',
                 'Crocidura leucodon', 'Crocidura russula', 'Crocidura suaveolens', 'Dryomys nitedula',
                 'Glis glis', 'Microtus agrestis', 'Micromys minutus', 'Mus musculus', 'Muscardinus avellanarius',
                 'Neomys fodiens', 'Rattus norvegicus', 'Rattus rattus', 'Sorex araneus', 'Sorex minutus'],
    "Treeshrews": ['Tupaia belangeri'],
    "Proboscideans": ['Elephas maximus']
}

## Conservation status plots

In [ ]:
import os
BEARER_TOKEN = os.environ.get('IUCN_REDLIST_API_TOKEN')  # Optional: never commit credentials


In [ ]:
iucn_status_code_to_description = {
    "NE": "NOT EVALUATED",
    "DD": "DATA DEFICIENT",
    "LC": "LEAST CONCERN",
    "NT": "NEAR THREATENED",
    "VU": "VULNERABLE",
    "EN": "ENDANGERED",
    "CR": "CRITICALLY ENDANGERED",
    "EW": "EXTINCT IN THE WILD",
    "EX": "EXTINCT",
}

In [ ]:
def get_latest_iucn_status(genus: str, species: str):
    """
    Query the IUCN API for the latest Red List status of a given species.

    Parameters:
        genus (str): Genus name (e.g., "Panthera")
        species (str): Species name (e.g., "leo")
        page (int): Pagination parameter (default: 0)
        per_page (int): Results per page (default: 100)

    Returns:
        dict: Contains species name, status, year, and source link
    """
    url = f"https://api.iucnredlist.org/api/v4/taxa/scientific_name?genus_name={genus}&species_name={species}"

    if not BEARER_TOKEN:
        return {'status': None, 'source': 'IUCN request skipped: no token configured'}
    headers = {
        "Authorization": f"Bearer {BEARER_TOKEN}",
        "Content-Type": "application/json"
    }

    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        return {"error": str(e)}

    if "assessments" not in data:
        return {"error": "No assessments found in response"}

    # Filter for latest global assessment
    latest = [
        a for a in data["assessments"]
        if a.get("latest") and any(scope["description"]["en"] == "Global" for scope in a.get("scopes", []))
    ]

    if not latest:
        return {"error": "No latest global assessment found"}

    record = latest[0]
    return {
        "scientific_name": data["taxon"]["scientific_name"],
        "status": record.get("red_list_category_code"),
        "year": record.get("year_published"),
        "source": record.get("url")
    }

In [ ]:
species_list_with_iucn_status = {}
for order, species_lst in species_by_order.items():
    species_list_with_iucn_status[order] = []
    for sp in species_lst:
        splt_lst = sp.split()
        genus = splt_lst[0]
        species = splt_lst[1]
        result = get_latest_iucn_status(genus, species)
        result["genus"] = genus
        result["species"] = species
        result["order"] = order
        if "status" in result and result["status"] in iucn_status_code_to_description:
            result["status_desc"] = iucn_status_code_to_description[result["status"]]
        else:
            result["status_desc"] = "UNKNOWN"
            result["scientific_name"] = sp
        species_list_with_iucn_status[order].append(result)

In [ ]:
# Convert species_list_with_iucn_status to json

with open("species_list_with_iucn_status.json", "w") as f:
    json.dump(species_list_with_iucn_status, f, indent=4)

In [ ]:
species_data = [species_playload for order, species_list in species_list_with_iucn_status.items() for species_playload in species_list ]

In [ ]:
def create_status_frequency_histogram(data):
    # Define the exact conservation status order
    status_order = ['CRITICALLY ENDANGERED', 'ENDANGERED', 'VULNERABLE', 'NEAR THREATENED', 'LEAST CONCERN', 'UNKNOWN']

    # Define colors - same as previous plots
    status_colors = {
        'CRITICALLY ENDANGERED': '#8B0000',  # Dark red
        'ENDANGERED': '#DC143C',             # Crimson
        'VULNERABLE': '#FF6347',             # Tomato
        'NEAR THREATENED': '#FF7F7F',        # Light coral
        'LEAST CONCERN': '#FFB6C1',          # Light pink
        'UNKNOWN': '#808080'                 # Gray
    }

    # Count frequencies of each status
    status_counts = Counter([item.get('status_desc', 'UNKNOWN') for item in data])

    # Prepare data for plotting, maintaining the order
    statuses = []
    frequencies = []
    colors = []

    for status in status_order:
        count = status_counts.get(status, 0)
        statuses.append(status)
        frequencies.append(count)
        colors.append(status_colors[status])

    # Add any unexpected statuses that might be in the data
    for status, count in status_counts.items():
        if status not in status_order:
            statuses.append(status)
            frequencies.append(count)
            colors.append('#808080')  # Default gray for unexpected statuses

    # Create the histogram
    fig, ax = plt.subplots(figsize=(12, 8))

    bars = ax.bar(range(len(statuses)), frequencies, color=colors,
                  edgecolor='black', linewidth=1, alpha=0.8)

    # Customize the plot
    ax.set_xlabel('Conservation Status', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Species', fontsize=12, fontweight='bold')
    ax.set_title('Distribution of Species by Conservation Status',
                 fontsize=14, fontweight='bold', pad=20)

    # Set x-axis labels
    ax.set_xticks(range(len(statuses)))
    ax.set_xticklabels(statuses, rotation=45, ha='right', fontsize=10)

    # Add value labels on top of bars
    for i, (bar, freq) in enumerate(zip(bars, frequencies)):
        if freq > 0:  # Only show label if there are species in this category
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                   str(freq), ha='center', va='bottom', fontweight='bold', fontsize=11)

    # Add grid for easier reading
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)

    # Set y-axis to start from 0
    ax.set_ylim(0, max(frequencies) * 1.15 if frequencies else 1)

    # Add summary statistics as text box
    total_species = sum(frequencies)
    threatened_statuses = ['CRITICALLY ENDANGERED', 'ENDANGERED', 'VULNERABLE']
    threatened_count = sum([status_counts.get(status, 0) for status in threatened_statuses])

    stats_text = f"Total Species: {total_species}\nThreatened Species: {threatened_count}\nPercentage Threatened: {threatened_count/total_species*100:.1f}%" if total_species > 0 else "No data available"

    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

    # Adjust layout to prevent label cutoff
    plt.tight_layout()

    # Print summary to console
    print("Conservation Status Frequency:")
    print("-" * 40)
    for status, freq in zip(statuses, frequencies):
        if freq > 0:
            percentage = (freq / total_species * 100) if total_species > 0 else 0
            print(f"{status:20}: {freq:3d} ({percentage:5.1f}%)")
    print("-" * 40)
    print(f"{'Total':20}: {total_species:3d} (100.0%)")

    return fig, ax


In [ ]:
# fig1, ax1 = create_status_frequency_histogram(species_data)

In [ ]:
def create_stacked_histogram_by_order(data):
    # Define status and order lists
    status_order = ['CRITICALLY ENDANGERED', 'ENDANGERED', 'VULNERABLE', 'NEAR THREATENED', 'LEAST CONCERN', 'UNKNOWN']
    order_list = ['Chiroptera', 'Treeshrews', 'Artiodactyla', 'Primates', 'Carnivora', 'Proboscideans', 'Rodentia']

    status_colors = {
        'CRITICALLY ENDANGERED': '#8B0000',
        'ENDANGERED': '#DC143C',
        'VULNERABLE': '#FF6347',
        'NEAR THREATENED': '#FF7F7F',
        'LEAST CONCERN': '#FFB6C1',
        'UNKNOWN': '#808080'
    }

    # Create data matrix: orders x statuses
    data_matrix = np.zeros((len(order_list), len(status_order)))

    for i, order in enumerate(order_list):
        order_data = [item for item in data if item.get('order') == order]
        status_counts = Counter([item.get('status_desc', 'UNKNOWN') for item in order_data])

        for j, status in enumerate(status_order):
            data_matrix[i, j] = status_counts.get(status, 0)

    # Create stacked bar chart
    fig, ax = plt.subplots(figsize=(11, 7))

    bottom = np.zeros(len(order_list))
    bars = []

    for j, status in enumerate(status_order):
        color = status_colors[status]
        bar = ax.bar(range(len(order_list)), data_matrix[:, j],
                    bottom=bottom, label=status, color=color,
                    edgecolor='black', linewidth=0.5, alpha=0.8)
        bars.append(bar)
        bottom += data_matrix[:, j]

    # Customize plot
    ax.set_xlabel('Taxonomic Order', fontsize=12, fontweight='normal')
    ax.set_ylabel('Number of Species', fontsize=12, fontweight='normal')
    # ax.set_title('Species Distribution by Order and Conservation Status',
    #              fontsize=14, fontweight='normal', pad=20)

    ax.set_xticks(range(len(order_list)))
    ax.set_xticklabels(order_list, rotation=0, ha='center', fontsize=10)  # Changed ha='right' to ha='center'

    # Add legend
    ax.legend(title='Conservation Status', loc='upper right') # bbox_to_anchor=(1.05, 1)

    # Add grid
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)

    plt.tight_layout()

    return fig, ax


def create_stacked_histogram_by_order_wo_elephants(data, palette="blue"):
    import numpy as np
    import matplotlib.pyplot as plt
    from collections import Counter

    # Define status and order lists
    status_order = [
        'CRITICALLY ENDANGERED',
        'ENDANGERED',
        'VULNERABLE',
        'NEAR THREATENED',
        'LEAST CONCERN',
        'UNKNOWN'
    ]

    order_list = [
        'Chiroptera',
        'Treeshrews',
        'Artiodactyla',
        'Primates',
        'Carnivora',
        'Rodentia'
    ]

    # ---- Color palettes ----
    blue_palette = {
        'CRITICALLY ENDANGERED': '#08306B',
        'ENDANGERED': '#08519C',
        'VULNERABLE': '#2171B5',
        'NEAR THREATENED': '#4292C6',
        'LEAST CONCERN': '#6BAED6',
        'UNKNOWN': '#BDBDBD'
    }

    # Lancet-style (colorblind-safe, publication quality)
    lancet_palette = {
        'CRITICALLY ENDANGERED': '#00468B',
        'ENDANGERED': '#ED0000',
        'VULNERABLE': '#42B540',
        'NEAR THREATENED': '#0099B4',
        'LEAST CONCERN': '#925E9F',
        'UNKNOWN': '#ADB6B6'
    }

    if palette.lower() == "lancet":
        status_colors = lancet_palette
    else:
        status_colors = blue_palette

    # ---- Create data matrix: orders x statuses ----
    data_matrix = np.zeros((len(order_list), len(status_order)))

    for i, order in enumerate(order_list):
        order_data = [item for item in data if item.get('order') == order]
        status_counts = Counter(
            [item.get('status_desc', 'UNKNOWN') for item in order_data]
        )

        for j, status in enumerate(status_order):
            data_matrix[i, j] = status_counts.get(status, 0)

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=(11, 7))

    bottom = np.zeros(len(order_list))

    for j, status in enumerate(status_order):
        ax.bar(
            range(len(order_list)),
            data_matrix[:, j],
            bottom=bottom,
            label=status,
            color=status_colors[status],
            edgecolor='black',
            linewidth=0.5,
            alpha=0.85
        )
        bottom += data_matrix[:, j]

    ax.set_xlabel('Taxonomic Order', fontsize=12)
    ax.set_ylabel('Number of Species', fontsize=12)

    ax.set_xticks(range(len(order_list)))
    ax.set_xticklabels(order_list, fontsize=10)

    ax.legend(title='Conservation Status', loc='upper right')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)

    plt.tight_layout()
    return fig, ax


In [ ]:
fig2, ax2 = create_stacked_histogram_by_order_wo_elephants(species_data, palette="blue")

In [ ]:
fig2.savefig("stacked_histogram_blue.png", dpi=700, bbox_inches="tight")

In [ ]:
def create_status_frequency_histogram_horizontal(data):
    status_order = ['CRITICALLY ENDANGERED', 'ENDANGERED', 'VULNERABLE',
                    'NEAR THREATENED', 'LEAST CONCERN', 'UNKNOWN']

    status_colors = {
        'CRITICALLY ENDANGERED': '#8B0000',
        'ENDANGERED': '#DC143C',
        'VULNERABLE': '#FF6347',
        'NEAR THREATENED': '#FF7F7F',
        'LEAST CONCERN': '#FFB6C1',
        'UNKNOWN': '#808080'
    }

    status_counts = Counter([item.get('status_desc', 'UNKNOWN') for item in data])

    statuses, frequencies, colors = [], [], []
    for status in status_order:
        count = status_counts.get(status, 0)
        statuses.append(status)
        frequencies.append(count)
        colors.append(status_colors[status])

    for status, count in status_counts.items():
        if status not in status_order:
            statuses.append(status)
            frequencies.append(count)
            colors.append('#808080')

    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(range(len(statuses)), frequencies, color=colors,
                   edgecolor='black', linewidth=1, alpha=0.8, height=0.5)

    ax.set_ylabel('Conservation Status', fontsize=12, fontweight='normal')
    ax.set_xlabel('Number of Species', fontsize=12, fontweight='normal')
    ax.set_title('Distribution of Species by Conservation Status',
                 fontsize=14, fontweight='normal', pad=20)

    ax.set_yticks(range(len(statuses)))
    ax.set_yticklabels(statuses, fontsize=10)

    # Value labels inside/next to bars
    for bar, freq in zip(bars, frequencies):
        if freq > 0:
            ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                    str(freq), va='center', ha='left', fontweight='normal', fontsize=11)

    # ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)

    plt.tight_layout()
    return fig, ax

In [ ]:
# fig3, ax3 = create_status_frequency_histogram_horizontal(species_data)

In [ ]:
def create_species_cluster_plot(data):
    # Extract unique status descriptions and orders
    status_descs = list(set([item['status_desc'] for item in data if 'status_desc' in item]))
    orders = list(set([item['order'] for item in data if 'order' in item]))

    print(f"Found status descriptions: {status_descs}")
    print(f"Found orders: {orders}")

    # Define conservation status order from least to most concerning
    status_order = ['LEAST CONCERN', 'NEAR THREATENED', 'VULNERABLE', 'UNKNOWN', 'ENDANGERED', 'CRITICALLY ENDANGERED']

    # Filter to only include statuses that exist in the data, maintaining order
    existing_statuses = [status for status in status_order if status in status_descs]
    # Add any unexpected statuses that might be in the data
    for status in status_descs:
        if status not in existing_statuses:
            existing_statuses.append(status)

    # Define colors - shades of red from light to dark (least to most concerning)
    # UNKNOWN gets a neutral gray color
    status_colors = {}
    red_shades = ['#ffcccc', '#ff9999', '#ff6666', '#ff3333', '#cc0000']  # Light to dark red

    for i, status in enumerate(existing_statuses):
        if status == 'UNKNOWN':
            status_colors[status] = '#808080'  # Gray for unknown
        else:
            # Find position in concern hierarchy (excluding UNKNOWN)
            concern_statuses = [s for s in existing_statuses if s != 'UNKNOWN']
            if status in concern_statuses:
                concern_index = concern_statuses.index(status)
                color_index = min(concern_index, len(red_shades) - 1)
                status_colors[status] = red_shades[color_index]
            else:
                status_colors[status] = '#ff6666'  # Default red if not found

    # Define marker shapes for each order
    marker_shapes = ['o', 's', 'P', 'p', '*', 'X', 'v']
    order_markers = dict(zip(orders, marker_shapes[:len(orders)]))

    # Define cluster centers for each status in order of concern
    np.random.seed(42)  # For reproducible results
    cluster_centers = {}

    # Arrange clusters in a logical pattern based on conservation concern
    positions = [
        (-4, 2),   # LEAST CONCERN - top left (light red)
        (-2, 4),   # NEAR THREATENED - top center (light-medium red)
        (2, 3),    # VULNERABLE - top right (medium red)
        (0, -3),   # UNKNOWN - bottom center (gray)
        (4, 1),    # ENDANGERED - right (dark red)
        (3, -1)    # CRITICALLY ENDANGERED - bottom right (darkest red)
    ]

    for i, status in enumerate(existing_statuses):
        if i < len(positions):
            cluster_centers[status] = positions[i]
        else:
            # Fallback for any additional statuses
            cluster_centers[status] = (np.random.uniform(-5, 5), np.random.uniform(-5, 5))

    # Create the plot
    fig, ax = plt.subplots(figsize=(14, 10))

    # Group data by status for plotting (maintain conservation concern order)
    for status in existing_statuses:
        status_items = [item for item in data if item.get('status_desc') == status]

        if not status_items:
            continue

        center_x, center_y = cluster_centers[status]

        # Generate random positions around cluster center
        for item in status_items:
            # Random offset from cluster center
            offset_x = np.random.normal(0, 1.5)  # Standard deviation of 1.5
            offset_y = np.random.normal(0, 1.5)

            x = center_x + offset_x
            y = center_y + offset_y

            order = item.get('order', 'Unknown')
            marker = order_markers.get(order, 'o')
            color = status_colors[status]

            # Plot the point
            ax.scatter(x, y, c=[color], marker=marker, s=100,
                      edgecolors='black', linewidth=1, alpha=0.8)

            # Add text label
            scientific_name = item.get('scientific_name', 'Unknown species')
            ax.annotate(scientific_name, (x, y), xytext=(5, 5),
                       textcoords='offset points', fontsize=8,
                       ha='left', va='bottom')

    # Customize the plot
    # ax.set_xlabel('X Coordinate', fontsize=12)
    # ax.set_ylabel('Y Coordinate', fontsize=12)
    # ax.set_title('Species Clustering by Conservation Status', fontsize=14, fontweight='bold')
    # ax.grid(True, alpha=0.3)

    # Create legend for status (colors)
    status_legend_elements = []
    for status, color in status_colors.items():
        status_legend_elements.append(plt.Line2D([0], [0], marker='o', color='w',
                                                markerfacecolor=color, markersize=10,
                                                markeredgecolor='black', label=status))

    # Create legend for orders (markers)
    order_legend_elements = []
    for order, marker in order_markers.items():
        order_legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w',
                                               markerfacecolor='gray', markersize=10,
                                               markeredgecolor='black', label=order))

    # Add legends
    legend1 = ax.legend(handles=status_legend_elements, title='Conservation Status',
                       loc='upper left', bbox_to_anchor=(1.02, 1))
    legend2 = ax.legend(handles=order_legend_elements, title='Order',
                       loc='upper left', bbox_to_anchor=(1.02, 0.6))
    ax.add_artist(legend1)  # Add the first legend back
    ax.set_xticks([])
    ax.set_yticks([])


    # Adjust layout to prevent legend cutoff
    plt.tight_layout()
    plt.subplots_adjust(right=0.75)

    return fig, ax

# Create and display the plot
if __name__ == "__main__":
    fig, ax = create_species_cluster_plot(species_data)
    plt.show()

    # Optional: Save the plot
    # plt.savefig('species_clusters.png', dpi=300, bbox_inches='tight')

In [ ]:
def create_species_subplot_visualization(data):
    # Define status descriptions in order (for subplot arrangement)
    status_order = ['CRITICALLY ENDANGERED', 'ENDANGERED', 'VULNERABLE', 'NEAR THREATENED', 'LEAST CONCERN', 'UNKNOWN']

    # Define all possible orders
    all_orders = ['Chiroptera', 'Treeshrews', 'Artiodactyla', 'Primates', 'Carnivora', 'Proboscideans', 'Rodentia']

    # Extract actual orders from data
    actual_orders = list(set([item['order'] for item in data if 'order' in item]))
    # Combine with predefined orders, maintaining order
    orders = [order for order in all_orders if order in actual_orders]
    for order in actual_orders:
        if order not in orders:
            orders.append(order)

    print(f"Status descriptions: {status_order}")
    print(f"Orders found: {orders}")

    # Define colors - same red gradient as before
    status_colors = {
        'LEAST CONCERN': '#ffcccc',
        'NEAR THREATENED': '#ff9999',
        'VULNERABLE': '#ff6666',
        'ENDANGERED': '#ff3333',
        'CRITICALLY ENDANGERED': '#cc0000',
        'UNKNOWN': '#808080'
    }

    # Define marker shapes for each order
    marker_shapes = ['o', 's', 'P', 'p', '*', 'X', 'v']
    order_markers = dict(zip(orders, marker_shapes[:len(orders)]))

    # Create subplots in 2x3 grid
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()  # Flatten for easy indexing

    # Set random seed for reproducible results
    np.random.seed(42)

    # Plot each status in its own subplot
    for i, status in enumerate(status_order):
        ax = axes[i]

        # Filter data for this status
        status_items = [item for item in data if item.get('status_desc') == status]

        if status_items:
            # Generate random positions for species in this status
            for item in status_items:
                # Random position within subplot
                x = np.random.uniform(-3, 3)
                y = np.random.uniform(-3, 3)

                order = item.get('order', 'Unknown')
                marker = order_markers.get(order, 'o')
                color = status_colors.get(status, '#666666')

                # Plot the point
                ax.scatter(x, y, c=color, marker=marker, s=120,
                          edgecolors='black', linewidth=1.5, alpha=0.8)

                # Add text label
                scientific_name = item.get('scientific_name', 'Unknown species')
                ax.annotate(scientific_name, (x, y), xytext=(5, 5),
                           textcoords='offset points', fontsize=9,
                           ha='left', va='bottom')

        # Customize each subplot
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_title(f'{status} ({len(status_items)} species)', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')

        # Remove axis labels for cleaner look
        ax.set_xticks([])
        ax.set_yticks([])

    # Create legend for orders (markers) - place it outside the subplots
    order_legend_elements = []
    for order, marker in order_markers.items():
        order_legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w',
                                               markerfacecolor='gray', markersize=12,
                                               markeredgecolor='black', label=order))

    # Add legend below the subplots
    fig.legend(handles=order_legend_elements, title='Taxonomic Orders',
              loc='upper left', bbox_to_anchor=(0.95, 0.6), fontsize=10)

    # Adjust layout
    plt.tight_layout()
    plt.subplots_adjust(top=0.9, bottom=0.15, left=0.1)

    return fig, axes

In [ ]:
fig, axes = create_species_subplot_visualization(species_data)

# Optional: Save the plot
# plt.savefig('species_status_subplots.png', dpi=300, bbox_inches='tight')

In [ ]:
def create_species_subplot_visualization_with_sobol(data):
    # Define status descriptions in order (for subplot arrangement)
    status_order = ['CRITICALLY ENDANGERED', 'ENDANGERED', 'VULNERABLE', 'NEAR THREATENED', 'LEAST CONCERN', 'UNKNOWN']

    # Define all possible orders
    all_orders = ['Chiroptera', 'Treeshrews', 'Artiodactyla', 'Primates', 'Carnivora', 'Proboscideans', 'Rodentia']

    # Extract actual orders from data
    actual_orders = list(set([item['order'] for item in data if 'order' in item]))
    # Combine with predefined orders, maintaining order
    orders = [order for order in all_orders if order in actual_orders]
    for order in actual_orders:
        if order not in orders:
            orders.append(order)

    print(f"Status descriptions: {status_order}")
    print(f"Orders found: {orders}")

    # Define colors - same red gradient as before
    status_colors = {
        'LEAST CONCERN': '#ffcccc',
        'NEAR THREATENED': '#ff9999',
        'VULNERABLE': '#ff6666',
        'ENDANGERED': '#ff3333',
        'CRITICALLY ENDANGERED': '#cc0000',
        'UNKNOWN': '#808080'
    }

    # Define marker shapes for each order
    marker_shapes = ['o', 's', 'P', 'p', '*', 'X', 'v']
    order_markers = dict(zip(orders, marker_shapes[:len(orders)]))

    # Create subplots in 2x3 grid
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()  # Flatten for easy indexing

    # Plot each status in its own subplot
    for i, status in enumerate(status_order):
        ax = axes[i]

        # Filter data for this status
        status_items = [item for item in data if item.get('status_desc') == status]

        if status_items:
            n_species = len(status_items)

            # Generate Sobol sequence for this subplot
            # We need 2D points (x, y coordinates)
            sobol_gen = qmc.Sobol(d=2, scramble=True, seed=42)
            sobol_points = sobol_gen.random(n_species)

            # Scale Sobol points from [0,1] to subplot range [a, b]
            a = -6
            b = 6
            scaled_points = sobol_points * (b - a + 1) + a

            # Plot each species using Sobol coordinates
            for j, item in enumerate(status_items):
                x, y = scaled_points[j]

                order = item.get('order', 'Unknown')
                marker = order_markers.get(order, 'o')
                color = status_colors.get(status, '#666666')

                # Plot the point
                ax.scatter(x, y, c=color, marker=marker, s=120,
                          edgecolors='black', linewidth=1.5, alpha=0.8)

                # Add text label
                scientific_name = item.get('scientific_name', 'Unknown species')
                ax.annotate(scientific_name, (x, y), xytext=(5, 5),
                           textcoords='offset points', fontsize=7,
                           ha='left', va='bottom')

        # Customize each subplot
        ax.set_xlim(-8, 8)
        ax.set_ylim(-8, 8)
        ax.set_title(f'{status} ({len(status_items)} species)', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')

        # Remove axis labels for cleaner look
        ax.set_xticks([])
        ax.set_yticks([])

    # Create legend for orders (markers) - place it outside the subplots
    order_legend_elements = []
    for order, marker in order_markers.items():
        order_legend_elements.append(plt.Line2D([0], [0], marker=marker, color='w',
                                               markerfacecolor='gray', markersize=12,
                                               markeredgecolor='black', label=order))

    # Add legend below the subplots
    fig.legend(handles=order_legend_elements, title='Taxonomic Orders',
              loc='upper left', bbox_to_anchor=(0.95, 0.6), fontsize=10)

    # Set main title
    fig.suptitle('Species Distribution by Conservation Status', fontsize=16, fontweight='bold', y=0.95)

    # Adjust layout
    plt.tight_layout()
    plt.subplots_adjust(top=0.9, bottom=0.15, left=0.1, right=0.85)

    return fig, axes

In [ ]:
fig, axes = create_species_subplot_visualization_with_sobol(species_data)

In [ ]:
def create_order_subplot_visualization(data):
    # Define status descriptions and colors
    status_order = ['CRITICALLY ENDANGERED', 'ENDANGERED', 'VULNERABLE',
                    'NEAR THREATENED', 'LEAST CONCERN', 'UNKNOWN']

    status_colors = {
        'LEAST CONCERN': '#ffcccc',
        'NEAR THREATENED': '#ff9999',
        'VULNERABLE': '#ff6666',
        'ENDANGERED': '#ff3333',
        'CRITICALLY ENDANGERED': '#cc0000',
        'UNKNOWN': '#808080'
    }

    # Extract all unique orders
    orders = sorted(list(set([item['order'] for item in data if 'order' in item])))

    # Decide subplot grid size (dynamic based on number of orders)
    n_orders = len(orders)
    ncols = 3
    nrows = int(np.ceil(n_orders / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 6*nrows))
    axes = axes.flatten()

    # Fix random seed for reproducibility
    np.random.seed(42)

    for i, order in enumerate(orders):
        ax = axes[i]

        # Filter species by this order
        order_items = [item for item in data if item.get('order') == order]

        if order_items:
            for item in order_items:
                # Random scatter placement
                x = np.random.uniform(-3, 3)
                y = np.random.uniform(-3, 3)

                status = item.get('status_desc', 'UNKNOWN')
                color = status_colors.get(status, '#666666')

                # Plot
                ax.scatter(x, y, c=color, marker='o', s=120,
                           edgecolors='black', linewidth=1.2, alpha=0.85)

                # Add species name as label
                ax.annotate(item['scientific_name'], (x, y), xytext=(5, 5),
                            textcoords='offset points', fontsize=9,
                            ha='left', va='bottom')

        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_title(f'{order} ({len(order_items)} species)', fontsize=13, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)

    # Hide unused subplots if any
    for j in range(i+1, len(axes)):
        axes[j].axis('off')

    # Create legend for statuses
    status_legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=status_colors[s],
                   markersize=12, markeredgecolor='black', label=s)
        for s in status_order
    ]

    fig.legend(handles=status_legend_elements, title='Conservation Status',
               loc='upper left', bbox_to_anchor=(0.92, 0.6), fontsize=10)

    plt.tight_layout()
    plt.subplots_adjust(top=0.9, right=0.88)

    return fig, axes

In [ ]:
fig, axes = create_order_subplot_visualization(species_data)

## Zoonotic Reservoirs Plot

In [ ]:
url = ROOT / 'data' / 'processed' / 'zoonotic_reservoirs.csv'


In [ ]:
zoonotic_reservoirs_df = pd.read_csv(url)

zoonotic_reservoirs_df = zoonotic_reservoirs_df[["Species in review", "Order", "Virus", "Recipients"]]

zoonotic_reservoirs_df.Recipients = zoonotic_reservoirs_df.Recipients.map(
    {
        "Humans": "Transmission to humans observed",
        "No bat-to-human transmission observed": "No transmission to humans observed",
        "No rat-to-human transmission observed": "No transmission to humans observed",
        "unknown transmission to humans": "Unknown transmission to humans",
        "potential transmission to humans": "Potential transmission to humans",
        "bat corona virus potential transmission to humans": "Potential transmission to humans",
        "in captivity": "In captivity"
    }
)
zoonotic_reservoirs_df['Recipients'] = zoonotic_reservoirs_df['Recipients'].fillna("")
zoonotic_reservoirs_df['Virus'] = zoonotic_reservoirs_df['Virus'].fillna("")

In [ ]:
def standardize_transmission_status(recipients):
    """Standardize transmission status for consistent categorization."""
    if not recipients or recipients.strip() == "":
        return "Unknown/Unpublished"

    recipients_lower = recipients.lower()

    if "no transmission to humans observed" in recipients_lower:
        return "No Transmission Observed"
    elif "transmission to humans observed" in recipients_lower:
        return "Transmission Observed"
    elif "potential transmission to humans" in recipients_lower:
        return "Potential Transmission"
    elif "unknown transmission to humans" in recipients_lower:
        return "Unknown Transmission"
    elif "in captivity" in recipients_lower:
        return "Captivity Only"
    else:
        return "Unknown/Unpublished"

def standardize_virus_category(virus):
    """Group viruses into broader categories for visualization."""
    if not virus or virus.lower().strip() in ['unknown', 'unpublished', '']:
        return 'Unknown/Unpublished'

    virus_lower = virus.lower()

    # Rabies and Lyssaviruses
    if any(term in virus_lower for term in ['rabies', 'lyssavirus', 'eblv', 'bblv']):
        return 'Rabies/Lyssavirus'

    # Coronaviruses
    elif 'coronavirus' in virus_lower or 'mers' in virus_lower:
        return 'Coronavirus'

    # Hepatitis viruses
    elif 'hepatitis' in virus_lower:
        return 'Hepatitis'

    # Paramyxoviruses
    elif 'paramyxovirus' in virus_lower:
        return 'Paramyxovirus'

    # Hantaviruses
    elif any(term in virus_lower for term in ['hanta', 'dobv', 'tula', 'htnv']):
        return 'Hantavirus'

    # Filoviruses (Marburg, Ebola, etc.)
    elif any(term in virus_lower for term in ['maarburg', 'marburg', 'hendra', 'lassa']):
        return 'Filovirus/High Risk'

    # Primate viruses
    elif any(term in virus_lower for term in ['siv', 'sfv', 'simian']):
        return 'Primate Viruses'

    # Bacterial diseases
    elif any(term in virus_lower for term in ['bacterial', 'tuberculosis', 'leptospira', 'bartonella', 'leshmania']):
        return 'Bacterial Diseases'

    # Parasites
    elif 'parasites' in virus_lower:
        return 'Parasites'

    # Bornaviruses
    elif 'bornavirus' in virus_lower:
        return 'Bornavirus'

    # Circoviruses
    elif 'circovirus' in virus_lower:
        return 'Circovirus'

    # Reoviruses
    elif 'reovirus' in virus_lower:
        return 'Reovirus'

    # Multiple/different families
    elif 'different families' in virus_lower:
        return 'Multiple Families'

    # Other specific viruses
    elif 'kobuvirus' in virus_lower:
        return 'Other RNA Viruses'
    elif 'picorna' in virus_lower:
        return 'Other RNA Viruses'

    else:
        return 'Other/Unclassified'

In [ ]:
zoonotic_reservoirs_df['Recipients'] = zoonotic_reservoirs_df['Recipients'].apply(standardize_transmission_status)
zoonotic_reservoirs_df['Virus'] = zoonotic_reservoirs_df['Virus'].apply(standardize_virus_category)

In [ ]:
# 2. Sankey Diagram
def plot_sankey_diagram():
    """Create a Sankey diagram showing flow from Order → Virus → Transmission status."""

    # Count flows
    order_virus = zoonotic_reservoirs_df.groupby(['Order', 'Virus']).size().reset_index(name='count')
    virus_transmission = zoonotic_reservoirs_df.groupby(['Virus', 'Recipients']).size().reset_index(name='count')

    # Create node lists
    orders = zoonotic_reservoirs_df['Order'].unique().tolist()
    viruses = zoonotic_reservoirs_df['Virus'].unique().tolist()
    transmissions = zoonotic_reservoirs_df['Recipients'].unique().tolist()

    # Create node labels and indices
    all_nodes = orders + viruses + transmissions
    node_colors = (['lightblue'] * len(orders) +
                  ['lightgreen'] * len(viruses) +
                  ['lightcoral', 'lightyellow', 'lightgray'][:len(transmissions)])

    # Create links
    source = []
    target = []
    value = []

    # Order to Virus links
    for _, row in order_virus.iterrows():
        source.append(all_nodes.index(row['Order']))
        target.append(all_nodes.index(row['Virus']))
        value.append(row['count'])

    # Virus to Transmission links
    for _, row in virus_transmission.iterrows():
        source.append(all_nodes.index(row['Virus']))
        target.append(all_nodes.index(row['Recipients']))
        value.append(row['count'])

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_nodes,
            color=node_colors
        ),
        link=dict(
            source=source,
            target=target,
            value=value
        ))])

    fig.update_layout(
        title_text="Order → Virus → Transmission Status",
        title_x=0.5,
        font_size=15,
        height=600
    )

    return fig

In [ ]:
plot_sankey_diagram()

In [ ]:
# 3. Nested Donut Charts
def plot_nested_donut_charts():
    """Create nested donut charts showing orders and transmission status."""

    # Count by order (inner ring)
    order_counts = zoonotic_reservoirs_df['Order'].value_counts()

    # Count by order and transmission (outer ring)
    transmission_counts = zoonotic_reservoirs_df.groupby(['Order', 'Recipients']).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(12, 10))

    # Define colors
    order_colors = plt.cm.Set3(np.linspace(0, 1, len(order_counts)))
    transmission_colors = {
        'Transmission Observed': '#FF4500',
        'No Transmission': '#FFD700',
        'Unknown/Unpublished': '#CCCCCC'
    }

    # Inner donut (Orders)
    wedges1, texts1, autotexts1 = ax.pie(
        order_counts.values,
        labels=order_counts.index,
        colors=order_colors,
        radius=0.7,
        wedgeprops=dict(width=0.3, edgecolor='white'),
        autopct='%1.1f%%',
        pctdistance=0.55,
        startangle=90,           # 🔑 lock start angle
        counterclock=False       # 🔑 consistent direction
    )

    # Outer donut (Transmission status)
    outer_data = []
    outer_labels = []
    outer_colors = []

    for order in order_counts.index:
        for transmission in transmission_counts.columns:
            if transmission in transmission_counts.loc[order] and transmission_counts.loc[order, transmission] > 0:
                outer_data.append(transmission_counts.loc[order, transmission])
                outer_labels.append(f"{order}\n{transmission}")
                outer_colors.append(transmission_colors.get(transmission, '#CCCCCC'))

    wedges2, texts2, autotexts2 = ax.pie(
        outer_data,
        labels=outer_labels,
        colors=outer_colors,
        radius=1.0,
        wedgeprops=dict(width=0.3, edgecolor='white'),
        autopct='%1.0f',
        pctdistance=0.85,
        labeldistance=1.1,
        startangle=90,           # 🔑 same fixed angle
        counterclock=False       # 🔑 same direction
    )

    # Customize text
    for text in texts1:
        text.set_fontsize(10)
        text.set_fontweight('bold')

    for text in texts2:
        text.set_fontsize(8)

    for autotext in autotexts2:
        autotext.set_fontsize(8)
        autotext.set_color('white')
        autotext.set_fontweight('bold')

    # Add title
    # ax.set_title(
    #     'Species Distribution by Order and Transmission Status',
    #     fontsize=14,
    #     fontweight='bold',
    #     pad=20
    # )

    # Custom legend
    legend_elements = [
        plt.Line2D(
            [0], [0], marker='o', color='w',
            markerfacecolor=color, markersize=10, label=status
        )
        for status, color in transmission_colors.items()
    ]

    ax.legend(
        handles=legend_elements, title='Transmission Status',
        loc='center', bbox_to_anchor=(1.3, 0.5)
    )

    plt.tight_layout()

In [ ]:
# plot_nested_donut_charts()

In [ ]:
def plot_stacked_bar_detailed():
    """Create stacked bar chart with detailed breakdown and additional panels."""

    # Create subplot layout
    fig, ax = plt.subplots(figsize=(10, 6))

    transmission_pivot = zoonotic_reservoirs_df.groupby(['Order', 'Recipients']).size().unstack(fill_value=0)

    colors = {
        'Transmission Observed': '#fb9a99',      # pastel red
        'No Transmission Observed': '#a6d854',   # pastel green
        'Unknown/Unpublished': '#cccccc',        # light grey
        'Unknown Transmission': '#fdbf6f',       # pastel orange
        'Potential Transmission': '#fff2ae',     # pastel yellow
        'Captivity Only': '#80b1d3'              # pastel blue
    }

    # Ensure all transmission types are present
    # for trans_type in colors.keys():
    #     if trans_type not in transmission_pivot.columns:
    #         transmission_pivot[trans_type] = 0

    # Reorder columns
    transmission_pivot = transmission_pivot[[col for col in colors.keys() if col in transmission_pivot.columns]]

    transmission_pivot.plot(
        kind='barh',
        stacked=True,
        ax=ax,
        color=[colors[col] for col in transmission_pivot.columns],
        edgecolor='black',
        linewidth=0.5,
        alpha=0.9,
    )

    # ax.set_title('Species Count by Order and Transmission Status', fontsize=14, fontweight='bold')
    ax.set_ylabel('Taxonomic Order', fontsize=12, fontweight='normal')
    ax.set_xlabel('Number of Species', fontsize=12, fontweight='normal')
    ax.legend(loc='upper right') # bbox_to_anchor=(1.05, 1),

    # Add value labels on bars
    # for container in ax.containers:
    #     ax.bar_label(container, label_type='center', fmt='%g')

    plt.tight_layout()

In [ ]:
plot_stacked_bar_detailed()

In [ ]:
def plot_virus_density_by_order():
    """Create stacked bar chart with detailed breakdown and additional panels."""

    # Create subplot layout
    fig, ax = plt.subplots(figsize=(10, 6))

    virus_diversity = zoonotic_reservoirs_df.groupby('Order')['Virus'].nunique().sort_values(ascending=True)

    bars = ax.barh(
        virus_diversity.index,
        virus_diversity.values,
        color='lightseagreen',
        edgecolor='black',
        alpha=0.7
    )

    # ax.set_title('Virus Diversity by Order', fontsize=14, fontweight='normal')
    ax.set_xlabel('Number of Different Virus Categories', fontsize=12, fontweight='normal')
    ax.set_ylabel('Taxonomic Order', fontsize=12, fontweight='normal')

    # Add value labels on bars
    # for i, (bar, value) in enumerate(zip(bars, virus_diversity.values)):
    #     ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
    #             str(value), ha='left', va='center', fontweight='normal')

    plt.tight_layout()

In [ ]:
plot_virus_density_by_order()